# Notebook 03: The Forward & Backward Passes — Backpropagation from Scratch

---

## What This Notebook Covers

This notebook builds **backpropagation** — the algorithm that lets every neural network learn — from absolute scratch, then shows that what we built matches PyTorch exactly. We will:

1. **Build a network's forward pass by hand** — a linear layer, a ReLU, another linear layer, and an MSE loss.
2. **Derive and code the backward pass** — the gradient of the loss with respect to *every* weight, using only the chain rule.
3. **Verify against PyTorch autograd** — proof our hand-derived gradients are correct.
4. **Refactor into layer objects** — each layer becomes a class that knows its own forward and backward.
5. **Abstract into a `Module` base class**, then finally **swap in PyTorch's `nn.Module` + autograd** and see we rebuilt exactly that.

---

## Why backpropagation?

Training a neural network means **adjusting its weights so the loss goes down**. To adjust a weight sensibly we need to know one thing: *if I nudge this weight a little, does the loss go up or down, and by how much?* That number is the **partial derivative of the loss with respect to the weight** — its **gradient**.

A modern network has millions of weights buried inside many composed functions. Computing each gradient independently would be hopeless. **Backpropagation** is the insight that you can get *all* of them in a single sweep: do the forward pass once (saving intermediate values), then walk backwards from the loss, multiplying one **local derivative** at each step (the chain rule). The cost of all gradients is about the same as one forward pass.

Understanding this from scratch pays off everywhere: debugging a network that won't learn, writing a custom layer, reasoning about vanishing/exploding gradients, and — for anyone doing physics-based modelling — recognising backprop as **reverse-mode automatic differentiation**, the same idea as the *adjoint method* used for sensitivity analysis and data assimilation in climate models.

---

## Prerequisites

You should be comfortable with:
- **Matrix multiplication and broadcasting** (notebook 01) — the forward pass is `x @ w + b`.
- **Basic calculus** — what a derivative is, the power rule, and the chain rule (we re-derive what we need).
- **Python classes** — Part 3 onward refactors layers into objects.

---

## Key terms

| Term | Meaning |
|------|---------|
| **Forward pass** | Run the input through each layer to produce a prediction and a loss. |
| **Backward pass** | Run gradients from the loss back through each layer to every parameter. |
| **Gradient** `∂L/∂θ` | How much the loss `L` changes if parameter `θ` changes a little. |
| **Chain rule** | Derivative of composed functions = product of local derivatives along the path. |
| **Local derivative** | The derivative of a single operation, in isolation (e.g. `d(u²)/du = 2u`). |
| **Loss** | A single number measuring how wrong the predictions are (here, MSE). |

---

## The learning cycle (the big picture)

```
   ┌──────────┐   FORWARD (compute)   ┌──────────┐   ┌──────────┐
   │  INPUT   │ ────────────────────► │  OUTPUT  │──►│   LOSS   │
   │  (image) │                       │  (pred)  │   │  (error) │
   └──────────┘                       └──────────┘   └────┬─────┘
        ▲                                                 │
        │              BACKWARD (gradients)               │
        │   ◄─────────────────────────────────────────────┘
   ┌────┴─────┐
   │  UPDATE  │   weight ← weight − learning_rate × gradient
   └──────────┘   …repeat thousands of times
```

Everything below is just filling in the two arrows — **compute** (forward) and **gradients** (backward) — in code, by hand, and then proving PyTorch does the same thing.


In [ ]:
# ============================================================================
# IMPORTING REQUIRED LIBRARIES AND LOADING DATA
# ============================================================================
# Let's import all the tools we need and load the MNIST dataset.
# MNIST is a classic dataset of handwritten digits (0-9), perfect for learning!

# ------------------------------------------------------------------------------
# STANDARD PYTHON LIBRARIES
# ------------------------------------------------------------------------------
import pickle      # For loading serialized Python objects from files
import gzip        # For reading compressed .gz files
import math        # Mathematical functions (we'll use this for calculus)
import os          # Operating system functions (file paths, etc.)
import time        # For timing our code
import shutil      # File operations (copy, move, etc.)

# ------------------------------------------------------------------------------
# CORE NUMERICAL LIBRARIES
# ------------------------------------------------------------------------------
import torch                    # PyTorch - our main deep learning library
import matplotlib as mpl        # Plotting library configuration
import numpy as np              # NumPy - for numerical operations

# ------------------------------------------------------------------------------
# UTILITY IMPORTS
# ------------------------------------------------------------------------------
from pathlib import Path        # Modern way to handle file paths
from torch import tensor        # Shortcut to create tensors easily
from fastcore.test import test_close  # Utility to test if values are close

# ------------------------------------------------------------------------------
# SET RANDOM SEED FOR REPRODUCIBILITY
# ------------------------------------------------------------------------------
# This ensures we get the same "random" numbers every time we run the notebook
# Important for debugging and comparing results!
torch.manual_seed(42)

# ------------------------------------------------------------------------------
# DISPLAY SETTINGS
# ------------------------------------------------------------------------------
mpl.rcParams['image.cmap'] = 'gray'  # Display images in grayscale
torch.set_printoptions(precision=2, linewidth=125, sci_mode=False)  # Nice tensor printing
np.set_printoptions(precision=2, linewidth=125)  # Nice array printing

# ------------------------------------------------------------------------------
# LOADING THE MNIST DATASET
# ------------------------------------------------------------------------------
# MNIST contains:
#   - 60,000 training images (we use 50,000)
#   - 10,000 validation images
#   - Each image is 28×28 = 784 pixels
#   - Labels are digits 0-9

path_data = Path('data')              # Where our data is stored
path_gz = path_data/'mnist.pkl.gz'    # Path to the compressed MNIST file

# Load the pickled (serialized) data from the gzipped file
# The file contains three tuples: (training data, validation data, test data)
# Each tuple contains (images, labels)
with gzip.open(path_gz, 'rb') as f:
    ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')

# Convert NumPy arrays to PyTorch tensors
# map() applies the tensor() function to each array
x_train, y_train, x_valid, y_valid = map(tensor, [x_train, y_train, x_valid, y_valid])

# Let's verify what we loaded
print("Dataset loaded successfully!")
print(f"Training images shape: {x_train.shape}")   # Should be (50000, 784)
print(f"Training labels shape: {y_train.shape}")   # Should be (50000,)
print(f"Validation images shape: {x_valid.shape}") # Should be (10000, 784)
print(f"Validation labels shape: {y_valid.shape}") # Should be (10000,)

**What does the code above do?**

Sets up the workspace and loads **MNIST** (70,000 handwritten digits, 28×28 = 784 pixels each). The key lines:

| Line | Purpose |
|---|---|
| `torch.manual_seed(42)` | Make the 'random' weights reproducible so results match every run. |
| `pickle.load(gzip.open(...))` | Read the compressed dataset into NumPy arrays. |
| `map(tensor, [...])` | Convert each NumPy array to a PyTorch tensor. |

We end up with `x_train` of shape **(50000, 784)** and `y_train` of shape **(50000,)** — 50k flattened images and their digit labels. Those 784 inputs are what our first layer will consume.

---
## Part 1: Building a Neural Network from Scratch

In this section, we'll build a simple neural network **without using any PyTorch magic**. This helps us understand exactly what's happening inside.

### Our Network Architecture

We'll build a network with:
- **Input layer**: 784 neurons (one per pixel)
- **Hidden layer**: 50 neurons (with ReLU activation)
- **Output layer**: 1 neuron (for simplicity, we'll predict a single number)

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        OUR NEURAL NETWORK                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   INPUT LAYER          HIDDEN LAYER           OUTPUT LAYER                  │
│   (784 neurons)        (50 neurons)           (1 neuron)                    │
│                                                                             │
│      ○ ─────┐                                                               │
│      ○ ─────┼────────► ○ ─────┐                                             │
│      ○ ─────┤   w1     ○ ─────┼────────► ○                                  │
│      ○ ─────┼────────► ○ ─────┤   w2     (prediction)                       │
│      ⋮      │          ⋮      │                                             │
│      ○ ─────┘          ○ ─────┘                                             │
│                                                                             │
│   Each pixel        Each hidden         Final output                        │
│   value             neuron uses         combines all                        │
│                     ReLU activation     hidden neurons                      │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### What happens at each layer?

1. **Linear Layer 1**: `output = input × W1 + b1`  
   - Takes 784 inputs, produces 50 outputs
   - W1 has shape (784, 50), b1 has shape (50,)

2. **ReLU Activation**: `output = max(0, input)`  
   - Keeps positive values, sets negatives to zero
   - Adds non-linearity (crucial for learning complex patterns!)

3. **Linear Layer 2**: `output = input × W2 + b2`  
   - Takes 50 inputs, produces 1 output
   - W2 has shape (50, 1), b2 has shape (1,)

### Step 1: Understanding Our Data Dimensions

Before building the network, we need to know our data dimensions. This determines the size of our weight matrices.

In [ ]:
# ============================================================================
# UNDERSTANDING DATA DIMENSIONS
# ============================================================================

# Get the shape of our training data
# x_train.shape returns (rows, columns) = (samples, features)
n, m = x_train.shape

# n = number of training samples (50,000 images)
# m = number of features per sample (784 pixels per image = 28×28)

# c = number of classes (digits 0-9 = 10 classes)
# y_train.max() finds the highest label (9)
# Adding 1 gives us the count (0,1,2,...,9 = 10 classes)
c = y_train.max() + 1

# Display our dimensions
print(f"Number of training samples (n): {n}")
print(f"Number of features per sample (m): {m}")
print(f"Number of classes (c): {c}")
print()
print("This means:")
print(f"  - We have {n} images to train on")
print(f"  - Each image has {m} pixels (flattened from 28×28)")
print(f"  - We need to classify into {c} different digits")

In [ ]:
# ============================================================================
# CHOOSING THE HIDDEN LAYER SIZE
# ============================================================================

# 'nh' stands for "number of hidden" neurons
# This is a hyperparameter - we choose it ourselves!
# 
# Why 50?
# - Too few neurons → network can't learn complex patterns
# - Too many neurons → network may memorize training data (overfitting)
# - 50 is a reasonable starting point for a simple demo

nh = 50  # Number of neurons in our hidden layer

print(f"Hidden layer size: {nh} neurons")
print(f"This means our network will learn {nh} different 'features' of digits")

In [ ]:
# ============================================================================
# INITIALIZING WEIGHTS AND BIASES
# ============================================================================
# These are the LEARNABLE PARAMETERS of our network.
# The network will adjust these during training to make better predictions.
#
# WEIGHTS: Control how strongly each input affects each output
# BIASES: Offset values that shift the output

# ─────────────────────────────────────────────────────────────────────────────
# LAYER 1: Input (784) → Hidden (50)
# ─────────────────────────────────────────────────────────────────────────────

# w1: Weight matrix for first layer
# Shape: (m, nh) = (784, 50)
# - 784 rows (one per input feature/pixel)
# - 50 columns (one per hidden neuron)
# - Total: 784 × 50 = 39,200 learnable weights!
#
# torch.randn creates random numbers from a normal distribution (mean=0, std=1)
# Random initialization is important - if all weights were the same,
# all neurons would learn the same thing!
w1 = torch.randn(m, nh)

# b1: Bias vector for first layer
# Shape: (nh,) = (50,)
# One bias per hidden neuron
# Initialize to zero - this is a common practice
b1 = torch.zeros(nh)

# ─────────────────────────────────────────────────────────────────────────────
# LAYER 2: Hidden (50) → Output (1)
# ─────────────────────────────────────────────────────────────────────────────

# w2: Weight matrix for second layer
# Shape: (nh, 1) = (50, 1)
# - 50 rows (one per hidden neuron)
# - 1 column (our single output)
w2 = torch.randn(nh, 1)

# b2: Bias for output layer
# Shape: (1,)
# Just one bias for our single output
b2 = torch.zeros(1)

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY OF OUR PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
print("Network Parameters:")
print(f"  w1 shape: {w1.shape} → {w1.shape[0] * w1.shape[1]:,} parameters")
print(f"  b1 shape: {b1.shape} → {b1.shape[0]:,} parameters")
print(f"  w2 shape: {w2.shape} → {w2.shape[0] * w2.shape[1]:,} parameters")
print(f"  b2 shape: {b2.shape} → {b2.shape[0]:,} parameters")
print(f"  TOTAL: {w1.numel() + b1.numel() + w2.numel() + b2.numel():,} learnable parameters")

**What does the code above do?**

Creates the network's **learnable parameters** — the only things training will change:

| Param | Shape | Role |
|---|---|---|
| `w1` | (784, 50) | weights, input → hidden |
| `b1` | (50,) | bias for the 50 hidden units |
| `w2` | (50, 1) | weights, hidden → output |
| `b2` | (1,) | bias for the single output |

Weights are drawn from `randn` (random normal) so that different units start out computing *different* things — if every weight were identical, every hidden unit would stay identical forever. Biases start at zero. *(This particular `randn` initialisation is actually too large and will be fixed in notebook 11 — here we just need something to differentiate.)*

In [ ]:
# ============================================================================
# THE LINEAR LAYER (FULLY CONNECTED LAYER)
# ============================================================================
# This is the fundamental building block of neural networks!
#
# FORMULA: output = input @ weights + bias
#
# In math notation: y = Wx + b
#   - x is the input (shape: batch_size × input_features)
#   - W is the weight matrix (shape: input_features × output_features)
#   - b is the bias vector (shape: output_features)
#   - y is the output (shape: batch_size × output_features)
#
# The @ symbol is matrix multiplication in Python

def lin(x, w, b):
    """
    Compute a linear transformation: output = x @ w + b
    
    Args:
        x: Input tensor of shape (batch_size, input_features)
           Example: (10000, 784) for 10000 images with 784 pixels each
        
        w: Weight matrix of shape (input_features, output_features)
           Example: (784, 50) to go from 784 pixels to 50 hidden neurons
        
        b: Bias vector of shape (output_features,)
           Example: (50,) for 50 hidden neurons
    
    Returns:
        Output tensor of shape (batch_size, output_features)
        Example: (10000, 50) - predictions for 10000 images, 50 values each
    
    How it works:
        1. x @ w: Matrix multiplication
           - Each row of x is multiplied with each column of w
           - Result shape: (batch_size, output_features)
        
        2. + b: Add bias (broadcasts across all samples)
           - Same bias added to every sample
    
    Visualization (single sample):
    ──────────────────────────────
    Input x = [x1, x2, ..., x784]  (one image, 784 pixels)
    
    Output neuron 1 = x1×w11 + x2×w21 + ... + x784×w784,1 + b1
    Output neuron 2 = x1×w12 + x2×w22 + ... + x784×w784,2 + b2
    ...
    Output neuron 50 = x1×w1,50 + x2×w2,50 + ... + x784×w784,50 + b50
    """
    return x @ w + b

# Quick explanation of the math:
# If x has shape (10000, 784) and w has shape (784, 50):
#   - Each of 10000 samples gets transformed
#   - Each sample: 784 inputs → 50 outputs
#   - Each output is a weighted sum of all 784 inputs

print("Linear layer function defined!")
print("Formula: output = input @ weights + bias")

**What does the code above do?**

Defines the **linear (fully-connected) layer**: `lin(x, w, b) = x @ w + b`. This is the workhorse of the network. For a batch `x` of shape `(bs, in)` and weights `w` of shape `(in, out)`, the matrix product gives `(bs, out)`: each output is a weighted sum of all inputs, and `+ b` broadcasts the same bias across every row of the batch. Two of these (with a ReLU between) make up our whole network.

In [ ]:
# ============================================================================
# TESTING THE LINEAR LAYER
# ============================================================================
# Let's pass our validation data through the first linear layer

# Apply linear transformation: x_valid (10000, 784) @ w1 (784, 50) + b1 (50)
# Result: t has shape (10000, 50)
t = lin(x_valid, w1, b1)

print(f"Input shape: {x_valid.shape}")     # (10000, 784)
print(f"Weight shape: {w1.shape}")          # (784, 50)  
print(f"Bias shape: {b1.shape}")            # (50,)
print(f"Output shape: {t.shape}")           # (10000, 50)
print()
print("Each of 10,000 images now has 50 values (one per hidden neuron)")

In [ ]:
# ============================================================================
# THE ReLU ACTIVATION FUNCTION
# ============================================================================
# ReLU stands for "Rectified Linear Unit"
# It's the most commonly used activation function in modern neural networks.
#
# FORMULA: ReLU(x) = max(0, x)
#   - If x > 0, output = x (keep positive values)
#   - If x ≤ 0, output = 0 (zero out negative values)
#
# WHY DO WE NEED ACTIVATION FUNCTIONS?
# ────────────────────────────────────
# Without activation functions, stacking linear layers is useless!
# linear(linear(x)) = x @ W1 @ W2 = x @ (W1 @ W2) = x @ W3
# It's still just a linear transformation!
#
# ReLU adds NON-LINEARITY, allowing the network to learn complex patterns.
#
# Visual representation:
#
#   Output
#     │           ╱
#     │         ╱
#     │       ╱
#     │     ╱
#  ───┼───●────────── Input
#     │   0
#     │
#
# Everything negative becomes 0, positives pass through unchanged.

def relu(x):
    """
    Apply ReLU (Rectified Linear Unit) activation function.
    
    Args:
        x: Input tensor of any shape
    
    Returns:
        Tensor of same shape with negative values set to 0
    
    Implementation:
        clamp_min(0.) sets all values below 0 to 0
        - clamp_min is a PyTorch function that clips minimum values
        - The 0. ensures we get a float, not an integer
    """
    return x.clamp_min(0.)

# Demonstrate ReLU behavior
demo = torch.tensor([-3., -1., 0., 1., 3.])
print("ReLU demonstration:")
print(f"  Input:  {demo.tolist()}")
print(f"  Output: {relu(demo).tolist()}")
print()
print("Notice: -3 and -1 become 0, while 1 and 3 stay the same!")

**What does the code above do?**

Defines **ReLU** (Rectified Linear Unit): `relu(x) = max(0, x)` — keep positives, zero out negatives. It is applied element-wise, so the shape is unchanged. ReLU is what makes the network *non-linear*, which is the whole reason it can learn interesting functions (see the deep dive below).

# Deep Dive: why we need a non-linearity

---

## The problem ReLU solves

Suppose we skipped the activation and just stacked two linear layers:

```python
out = lin(lin(x, w1, b1), w2, b2)     # no ReLU in between
```

Algebraically that is

$$\text{out} = (x W_1 + b_1) W_2 + b_2 = x \,(W_1 W_2) + (b_1 W_2 + b_2) = x\,W' + b'.$$

The product `W₁W₂` is just *another matrix* `W'`, and `b₁W₂+b₂` is just another vector `b'`. So **two linear layers collapse into one linear layer.** No matter how many you stack, a pile of linear layers can only ever draw a straight line / flat plane — it can never bend to fit a curve, an XOR, or a digit.

## What ReLU adds

Inserting `relu` between the layers breaks that collapse:

```python
out = lin(relu(lin(x, w1, b1)), w2, b2)   # the bend lives in relu
```

`relu` introduces a **kink** at 0. Each hidden unit is a line that gets clamped to zero on one side; summing 50 such clamped lines lets the network build arbitrarily bumpy, curved decision surfaces. This is the practical face of the **universal approximation theorem**: linear layers + a simple non-linearity can approximate essentially any function, given enough units.

ReLU specifically is popular because it is dirt cheap (`max(0,x)`) and its gradient is trivial — **exactly 1 for positive inputs and 0 for negative ones**. That clean 0/1 gradient is what we will multiply by in the backward pass. Play with it below: drag `x` and watch both the function and its gradient.

### 🎮 Interactive: ReLU as an on/off gate

Drag **x**. The left panel is `ReLU(x)`; the right is its gradient. Notice the gradient is a flat **1** wherever the input is positive and a flat **0** wherever it's negative — ReLU either passes a signal through untouched or blocks it completely. That 0/1 mask is exactly what backprop multiplies by later.

In [ ]:
# ============================================================================
# INTERACTIVE (embedded figure) -- run this cell. Self-contained iframe;
# works in Jupyter, Colab, and the exported HTML. Re-run to reset it.
# ============================================================================
from IPython.display import HTML
HTML('<iframe src="data:text/html;base64,PCFET0NUWVBFIGh0bWw+PGh0bWwgbGFuZz0iZW4iPjxoZWFkPjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+UmVMVTwvdGl0bGU+CjxzdHlsZT4KICA6cm9vdHstLWluazojMjUzMjRhOy0tc29mdDojNWE2Yjg2Oy0tbGluZTojZGRlNWYyOy0taW46I2Y2OTIxZTstLW91dDojMTZhM2EzOy0tYWM6IzdiNWNkNjstLWJnOiNlZWYzZmI7fQogICp7Ym94LXNpemluZzpib3JkZXItYm94O31odG1sLGJvZHl7bWFyZ2luOjA7Zm9udC1mYW1pbHk6IlNlZ29lIFVJIixzeXN0ZW0tdWksc2Fucy1zZXJpZjtjb2xvcjp2YXIoLS1pbmspO30KICBib2R5e2JhY2tncm91bmQ6cmFkaWFsLWdyYWRpZW50KDkwMHB4IDQwMHB4IGF0IDMwJSAtMTAlLCNmM2Y3ZmYsdmFyKC0tYmcpKSx2YXIoLS1iZyk7cGFkZGluZzoxNHB4O30KICBoMXtmb250LXNpemU6MTdweDttYXJnaW46MCAwIDJweDt0ZXh0LWFsaWduOmNlbnRlcjt9CiAgLnN1Ynt0ZXh0LWFsaWduOmNlbnRlcjtjb2xvcjp2YXIoLS1zb2Z0KTtmb250LXNpemU6MTJweDttYXJnaW46MCAwIDEwcHg7fQogIC53cmFwe2Rpc3BsYXk6ZmxleDtnYXA6MTRweDtqdXN0aWZ5LWNvbnRlbnQ6Y2VudGVyO2FsaWduLWl0ZW1zOnN0cmV0Y2g7fQogIC5jYXJke2JhY2tncm91bmQ6I2ZmZjtib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpO2JvcmRlci1yYWRpdXM6MTJweDtwYWRkaW5nOjEwcHg7Ym94LXNoYWRvdzowIDZweCAxNnB4IHJnYmEoMTIzLDkyLDIxNCwuMDgpO2ZsZXg6MSAxIDA7bWluLXdpZHRoOjA7bWF4LXdpZHRoOjMzMHB4O30KICBjYW52YXN7ZGlzcGxheTpibG9jaztiYWNrZ3JvdW5kOiNmYmZkZmY7Ym9yZGVyLXJhZGl1czo4cHg7Ym9yZGVyOjFweCBzb2xpZCB2YXIoLS1saW5lKTt3aWR0aDoxMDAlO2hlaWdodDphdXRvO30KICBAbWVkaWEobWF4LXdpZHRoOjUyMHB4KXsud3JhcHtmbGV4LWRpcmVjdGlvbjpjb2x1bW47fS5jYXJke21heC13aWR0aDoxMDAlO319CiAgLmNhcHt0ZXh0LWFsaWduOmNlbnRlcjtmb250LXNpemU6MTJweDtmb250LXdlaWdodDo3MDA7bWFyZ2luOjRweCAwO30KICAucmVhZG91dHtiYWNrZ3JvdW5kOiNmNGY3ZmQ7Ym9yZGVyLXJhZGl1czo4cHg7cGFkZGluZzo4cHggMTFweDtmb250LWZhbWlseTptb25vc3BhY2U7Zm9udC1zaXplOjEzcHg7bWFyZ2luLXRvcDo4cHg7bGluZS1oZWlnaHQ6MS43O30KICAudmFse2NvbG9yOnZhcigtLWluKTtmb250LXdlaWdodDo3MDA7fS5ne2NvbG9yOnZhcigtLW91dCk7Zm9udC13ZWlnaHQ6NzAwO30KICAucm93e2Rpc3BsYXk6ZmxleDtnYXA6OHB4O2FsaWduLWl0ZW1zOmNlbnRlcjtqdXN0aWZ5LWNvbnRlbnQ6Y2VudGVyO21hcmdpbjoxMnB4IDAgMnB4O2ZvbnQtc2l6ZToxM3B4O30KICBpbnB1dFt0eXBlPXJhbmdlXXt3aWR0aDoyMzBweDttYXgtd2lkdGg6NzB2dzthY2NlbnQtY29sb3I6dmFyKC0taW4pO30KICAubm90ZXt0ZXh0LWFsaWduOmNlbnRlcjtmb250LXNpemU6MTEuNXB4O2NvbG9yOnZhcigtLXNvZnQpO21hcmdpbi10b3A6OHB4O21heC13aWR0aDo1NjBweDttYXJnaW4tbGVmdDphdXRvO21hcmdpbi1yaWdodDphdXRvO2xpbmUtaGVpZ2h0OjEuNTt9Cjwvc3R5bGU+PC9oZWFkPjxib2R5Pgo8aDE+UmVMVSBhbmQgaXRzIGdyYWRpZW50OiBhbiBvbi9vZmYgZ2F0ZTwvaDE+CjxwIGNsYXNzPSJzdWIiPlJlTFUoeCkgPSBtYXgoMCwgeCkuIERyYWcgeC4gTm90aWNlIHRoZSBncmFkaWVudCBpcyBqdXN0IDxiPjEgd2hlbiB4Jmd0OzA8L2I+IGFuZCA8Yj4wIHdoZW4geCZsdDswPC9iPiDigJQgaXQgcGFzc2VzIHRoZSBzaWduYWwgb3IgYmxvY2tzIGl0LjwvcD4KPGRpdiBjbGFzcz0id3JhcCI+CiAgPGRpdiBjbGFzcz0iY2FyZCI+PGRpdiBjbGFzcz0iY2FwIiBzdHlsZT0iY29sb3I6dmFyKC0taW4pIj5SZUxVKHgpID0gbWF4KDAsIHgpPC9kaXY+PGNhbnZhcyBpZD0iY2YiIHdpZHRoPSIzMDAiIGhlaWdodD0iMjIwIj48L2NhbnZhcz48L2Rpdj4KICA8ZGl2IGNsYXNzPSJjYXJkIj48ZGl2IGNsYXNzPSJjYXAiIHN0eWxlPSJjb2xvcjp2YXIoLS1vdXQpIj5ncmFkaWVudCBSZUxVJyh4KTwvZGl2PjxjYW52YXMgaWQ9ImNnIiB3aWR0aD0iMzAwIiBoZWlnaHQ9IjIyMCI+PC9jYW52YXM+PC9kaXY+CjwvZGl2Pgo8ZGl2IGNsYXNzPSJyb3ciPnggPSA8aW5wdXQgaWQ9IngiIHR5cGU9InJhbmdlIiBtaW49Ii00IiBtYXg9IjQiIHN0ZXA9IjAuMSIgdmFsdWU9IjEuMyI+PHNwYW4gaWQ9Inh2IiBzdHlsZT0iZm9udC1mYW1pbHk6bW9ub3NwYWNlO2NvbG9yOnZhcigtLWluKTtmb250LXdlaWdodDo3MDAiPjEuMzwvc3Bhbj48L2Rpdj4KPGRpdiBjbGFzcz0icmVhZG91dCIgc3R5bGU9Im1heC13aWR0aDo0MjBweDttYXJnaW4tbGVmdDphdXRvO21hcmdpbi1yaWdodDphdXRvOyI+CiAgUmVMVSg8c3BhbiBjbGFzcz0idmFsIiBpZD0icngiPjEuMzwvc3Bhbj4pID0gPHNwYW4gY2xhc3M9InZhbCIgaWQ9InJ5Ij4xLjMwPC9zcGFuPjxicj4KICBSZUxVJyg8c3BhbiBjbGFzcz0idmFsIiBpZD0icngyIj4xLjM8L3NwYW4+KSA9IDxzcGFuIGNsYXNzPSJnIiBpZD0icmciPjE8L3NwYW4+ICZuYnNwOyA8c3BhbiBpZD0id2h5IiBzdHlsZT0iY29sb3I6dmFyKC0tc29mdCkiPjwvc3Bhbj4KPC9kaXY+CjxwIGNsYXNzPSJub3RlIj5JbiBiYWNrcHJvcCwgdGhlIGdyYWRpZW50IGZsb3dpbmcgYmFjayB0aHJvdWdoIFJlTFUgaXMgbXVsdGlwbGllZCBieSB0aGlzIDAtb3ItMSBtYXNrOiA8Y29kZT5sMS5nID0gKGwxICZndDsgMCkuZmxvYXQoKSAqIGwyLmc8L2NvZGU+LiBXaGVyZSB0aGUgdW5pdCB3YXMgIm9mZiIgKOKJpDApLCBubyBncmFkaWVudCBmbG93cyBiYWNrIHRocm91Z2ggaXQuPC9wPgo8c2NyaXB0Pgpjb25zdCAkPWlkPT5kb2N1bWVudC5nZXRFbGVtZW50QnlJZChpZCk7CmZ1bmN0aW9uIGRyYXcoY3YsZm4sY29sb3IseCl7CiAgY29uc3QgY3R4PWN2LmdldENvbnRleHQoJzJkJyksVz1jdi53aWR0aCxIPWN2LmhlaWdodCxwYWQ9MjI7CiAgY3R4LmNsZWFyUmVjdCgwLDAsVyxIKTsKICBjb25zdCB4bWluPS00LHhtYXg9NCx5bWluPS0wLjUseW1heD00OwogIGNvbnN0IFg9dj0+cGFkKyh2LXhtaW4pLyh4bWF4LXhtaW4pKihXLTIqcGFkKTsKICBjb25zdCBZPXY9PkgtcGFkLSh2LXltaW4pLyh5bWF4LXltaW4pKihILTIqcGFkKTsKICAvLyBheGVzCiAgY3R4LnN0cm9rZVN0eWxlPScjZTNlOWY0JztjdHgubGluZVdpZHRoPTE7CiAgY3R4LmJlZ2luUGF0aCgpO2N0eC5tb3ZlVG8oWCgwKSxwYWQpO2N0eC5saW5lVG8oWCgwKSxILXBhZCk7Y3R4Lm1vdmVUbyhwYWQsWSgwKSk7Y3R4LmxpbmVUbyhXLXBhZCxZKDApKTtjdHguc3Ryb2tlKCk7CiAgY3R4LmZpbGxTdHlsZT0nIzlmYjBjYyc7Y3R4LmZvbnQ9JzEwcHggbW9ub3NwYWNlJztjdHguZmlsbFRleHQoJzAnLFgoMCkrMyxZKDApKzEyKTsKICAvLyBjdXJ2ZQogIGN0eC5zdHJva2VTdHlsZT1jb2xvcjtjdHgubGluZVdpZHRoPTM7Y3R4LmJlZ2luUGF0aCgpO2xldCBzdGFydGVkPWZhbHNlOwogIGZvcihsZXQgcHg9cGFkO3B4PD1XLXBhZDtweCsrKXtjb25zdCB4dj14bWluKyhweC1wYWQpLyhXLTIqcGFkKSooeG1heC14bWluKTtjb25zdCB5dj1mbih4dik7CiAgICBjb25zdCBweT1ZKHl2KTsgaWYoeXY8eW1pbi0xfHx5dj55bWF4KzEpe3N0YXJ0ZWQ9ZmFsc2U7Y29udGludWU7fQogICAgaWYoIXN0YXJ0ZWQpe2N0eC5tb3ZlVG8ocHgscHkpO3N0YXJ0ZWQ9dHJ1ZTt9ZWxzZSBjdHgubGluZVRvKHB4LHB5KTt9CiAgY3R4LnN0cm9rZSgpOwogIC8vIHBvaW50CiAgY29uc3QgeXY9Zm4oeCk7Y3R4LmZpbGxTdHlsZT1jb2xvcjtjdHguYmVnaW5QYXRoKCk7Y3R4LmFyYyhYKHgpLFkoeXYpLDUsMCw3KTtjdHguZmlsbCgpOwogIGN0eC5zdHJva2VTdHlsZT0nI2ZmZic7Y3R4LmxpbmVXaWR0aD0yO2N0eC5zdHJva2UoKTsKICAvLyBkYXNoZWQgZ3VpZGUKICBjdHguc3Ryb2tlU3R5bGU9Y29sb3IrJzY2JztjdHguc2V0TGluZURhc2goWzQsM10pO2N0eC5saW5lV2lkdGg9MS41OwogIGN0eC5iZWdpblBhdGgoKTtjdHgubW92ZVRvKFgoeCksWSgwKSk7Y3R4LmxpbmVUbyhYKHgpLFkoeXYpKTtjdHguc3Ryb2tlKCk7Y3R4LnNldExpbmVEYXNoKFtdKTsKfQpjb25zdCByZWx1PXY9Pk1hdGgubWF4KDAsdiksIGRyZWx1PXY9PnY+MD8xOjA7CmZ1bmN0aW9uIHJlbmRlcigpe2NvbnN0IHg9KyQoJ3gnKS52YWx1ZTsKICAkKCd4dicpLnRleHRDb250ZW50PXgudG9GaXhlZCgxKTskKCdyeCcpLnRleHRDb250ZW50PXgudG9GaXhlZCgxKTskKCdyeDInKS50ZXh0Q29udGVudD14LnRvRml4ZWQoMSk7CiAgJCgncnknKS50ZXh0Q29udGVudD1yZWx1KHgpLnRvRml4ZWQoMik7JCgncmcnKS50ZXh0Q29udGVudD1kcmVsdSh4KTsKICAkKCd3aHknKS50ZXh0Q29udGVudCA9IHg+MD8nKHg+MCDihpIgcGFzcyknOih4PDA/Jyh4PDAg4oaSIGJsb2NrKSc6JyhhdCAwKScpOwogIGRyYXcoJCgnY2YnKSxyZWx1LCcjZjY5MjFlJyx4KTtkcmF3KCQoJ2NnJyksdj0+ZHJlbHUodiksJyMxNmEzYTMnLHgpO30KJCgneCcpLm9uaW5wdXQ9cmVuZGVyO3JlbmRlcigpOwo8L3NjcmlwdD48L2JvZHk+PC9odG1sPgo=" style="width:100%; height:520px; border:1px solid #dde5f2; border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);" loading="lazy" title="ReLU and its gradient"></iframe>')


In [ ]:
# ============================================================================
# APPLYING ReLU TO OUR LINEAR OUTPUT
# ============================================================================
# Remember: t = lin(x_valid, w1, b1) from earlier
# Now we apply ReLU to introduce non-linearity

t = relu(t)

# Display the output
print(f"After ReLU, shape is still: {t.shape}")
print()
print("First few values of first sample:")
print(t[0, :10])  # First 10 values of first sample
print()
print("Notice: All values are ≥ 0 (ReLU zeroed out negatives)")

In [ ]:
# ============================================================================
# THE COMPLETE FORWARD PASS (MODEL)
# ============================================================================
# Now let's combine everything into a single model function.
# This is called the "forward pass" - data flows forward through the network.

def model(xb):
    """
    Complete forward pass through our neural network.
    
    Args:
        xb: Input batch of images, shape (batch_size, 784)
            - batch_size: Number of images to process at once
            - 784: Number of pixels per image (28×28 flattened)
    
    Returns:
        Predictions, shape (batch_size, 1)
        - One prediction value per image
    
    Network Flow:
    ─────────────
    
    INPUT (batch_size, 784)
           │
           ▼
    ┌─────────────────────┐
    │ LINEAR LAYER 1      │  l1 = xb @ w1 + b1
    │ (784 → 50)          │  Shape: (batch_size, 50)
    └─────────────────────┘
           │
           ▼
    ┌─────────────────────┐
    │ ReLU ACTIVATION     │  l2 = max(0, l1)
    │                     │  Shape: (batch_size, 50)
    └─────────────────────┘
           │
           ▼
    ┌─────────────────────┐
    │ LINEAR LAYER 2      │  output = l2 @ w2 + b2
    │ (50 → 1)            │  Shape: (batch_size, 1)
    └─────────────────────┘
           │
           ▼
    OUTPUT (batch_size, 1)
    """
    # Step 1: First linear layer (784 inputs → 50 outputs)
    l1 = lin(xb, w1, b1)   # Shape: (batch_size, 50)
    
    # Step 2: ReLU activation (adds non-linearity)
    l2 = relu(l1)          # Shape: (batch_size, 50)
    
    # Step 3: Second linear layer (50 inputs → 1 output)
    output = lin(l2, w2, b2)  # Shape: (batch_size, 1)
    
    return output

print("Model (forward pass) function defined!")
print("Flow: Input → Linear → ReLU → Linear → Output")

In [ ]:
# ============================================================================
# TESTING THE FORWARD PASS
# ============================================================================
# Let's run the full forward pass on our validation data

res = model(x_valid)

print(f"Input shape: {x_valid.shape}")   # (10000, 784)
print(f"Output shape: {res.shape}")       # (10000, 1)
print()
print("Each of 10,000 images now has 1 prediction value!")
print()
print("First 5 predictions:")
print(res[:5].squeeze())  # Remove the trailing dimension for cleaner display

---
### The Loss Function: Measuring How Wrong We Are

A **loss function** (also called cost function or objective function) measures the difference between our predictions and the true values. The goal of training is to **minimize this loss**.

### Mean Squared Error (MSE)

We'll use **MSE** for simplicity:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

Where:
- $\hat{y}_i$ is our prediction for sample $i$
- $y_i$ is the true label for sample $i$
- $n$ is the number of samples

**Why squared?**
- Squaring makes all errors positive (no cancellation)
- Squaring penalizes large errors more than small ones
- The derivative is simple (important for backprop!)

**Note**: MSE isn't ideal for classification (cross-entropy is better), but we use it here because its gradient is simpler to understand.

**Important Note**: MSE is not the best loss function for classification (where we want to pick one of 10 digits). Cross-entropy loss is much better for that. We use MSE here because:
1. Its derivative is very simple (great for learning backprop!)
2. It still works for understanding the concepts
3. We'll use proper loss functions in later notebooks

In [ ]:
# ============================================================================
# UNDERSTANDING SHAPE COMPATIBILITY
# ============================================================================
# Before computing loss, we need to make sure our prediction and target
# shapes are compatible for subtraction.

print("Prediction shape:", res.shape)      # (10000, 1)
print("Target shape:", y_valid.shape)       # (10000,)
print()
print("Problem: res has shape (10000, 1) but y_valid has shape (10000,)")
print("This is a 2D tensor vs a 1D tensor - they don't match!")

In [ ]:
# ============================================================================
# THE BROADCASTING PROBLEM
# ============================================================================
# What happens if we try to subtract them directly?

result = (res - y_valid)
print(f"(res - y_valid).shape = {result.shape}")
print()
print("WRONG! We got (10000, 10000) instead of (10000,)")
print()
print("What happened?")
print("  res has shape (10000, 1)")
print("  y_valid has shape (10000,)")
print("  Broadcasting treats (10000,) as (1, 10000)")
print("  (10000, 1) - (1, 10000) broadcasts to (10000, 10000)!")
print()
print("This creates a matrix of 100 million differences - not what we want!")

### Solution: Remove the Trailing Dimension

We need to convert `res` from shape `(10000, 1)` to shape `(10000,)`.

There are two common ways to do this:
1. **Indexing**: `res[:, 0]` - take all rows, first column only
2. **Squeeze**: `res.squeeze()` - remove all dimensions of size 1

In [ ]:
# ============================================================================
# METHOD 1: Using indexing [:, 0]
# ============================================================================
# res[:, 0] means: "all rows (:), first column (0)"
# This converts (10000, 1) → (10000,)

print(f"Original res shape: {res.shape}")          # (10000, 1)
print(f"After res[:, 0]: {res[:, 0].shape}")        # (10000,)

In [ ]:
# ============================================================================
# METHOD 2: Using squeeze()
# ============================================================================
# squeeze() removes all dimensions of size 1
# (10000, 1) → (10000,)
# (1, 10000, 1) → (10000,)

print(f"Original res shape: {res.shape}")          # (10000, 1)
print(f"After squeeze(): {res.squeeze().shape}")    # (10000,)

In [ ]:
# ============================================================================
# VERIFYING THE SUBTRACTION WORKS NOW
# ============================================================================
# Now let's check that subtraction gives us the right shape

correct_diff = res[:, 0] - y_valid
print(f"(res[:, 0] - y_valid).shape = {correct_diff.shape}")
print()
print("Now we get (10000,) - one difference per sample. This is correct!")

In [ ]:
# ============================================================================
# CONVERTING LABELS TO FLOAT
# ============================================================================
# Our labels are integers (0, 1, 2, ..., 9)
# For MSE calculation, we need them as floats (0.0, 1.0, 2.0, ..., 9.0)
# PyTorch prefers matching types for operations

y_train, y_valid = y_train.float(), y_valid.float()

print(f"y_train dtype: {y_train.dtype}")  # Should be torch.float32
print(f"y_valid dtype: {y_valid.dtype}")  # Should be torch.float32

# Now run the model on training data
preds = model(x_train)
print(f"\nTraining predictions shape: {preds.shape}")

In [ ]:
# ============================================================================
# THE MEAN SQUARED ERROR (MSE) LOSS FUNCTION
# ============================================================================
# MSE = (1/n) × Σ(prediction - target)²
#
# Steps:
#   1. output[:, 0] - targ: Compute the difference (error) for each sample
#   2. .pow(2): Square each error (makes all positive, penalizes large errors)
#   3. .mean(): Average across all samples

def mse(output, targ):
    """
    Compute Mean Squared Error loss.
    
    Args:
        output: Model predictions, shape (batch_size, 1)
                Each row is the model's prediction for one sample
        
        targ: True labels, shape (batch_size,)
              The correct answer for each sample
    
    Returns:
        A single number: the average squared error across all samples
    
    Formula:
        MSE = (1/n) × Σᵢ(outputᵢ - targᵢ)²
    
    Why each step:
        output[:, 0]:  Convert (n, 1) → (n,) for compatible subtraction
        - targ:        Compute error (prediction - truth)
        .pow(2):       Square to make positive and penalize large errors
        .mean():       Average to get a single loss value
    """
    return (output[:, 0] - targ).pow(2).mean()

print("MSE loss function defined!")
print("Formula: MSE = mean( (prediction - target)² )")

**What does the code above do?**

Defines the **Mean Squared Error** loss: `mse(out, targ) = (out[:,0] - targ).pow(2).mean()`. Three steps — subtract to get the per-sample error, square it (all-positive, and big errors hurt more), then average to a single number. `out[:,0]` drops the trailing size-1 dimension so a `(bs,1)` prediction lines up with a `(bs,)` target (the broadcasting trap shown two cells above). This one scalar is what the backward pass will differentiate.

### 🎮 Interactive: the loss bowl and its slope

Drag the **prediction** (and the **target**). The orange curve is `(pred − target)²`; the teal line is its slope, the gradient `2(pred − target)`. The slope is **big when you're far from the target, zero at the bottom**, and its **sign tells you which way to move** the prediction. The backward pass starts from exactly this number.

In [ ]:
# ============================================================================
# INTERACTIVE (embedded figure) -- run this cell. Self-contained iframe;
# works in Jupyter, Colab, and the exported HTML. Re-run to reset it.
# ============================================================================
from IPython.display import HTML
HTML('<iframe src="data:text/html;base64,PCFET0NUWVBFIGh0bWw+PGh0bWwgbGFuZz0iZW4iPjxoZWFkPjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+TVNFPC90aXRsZT4KPHN0eWxlPgogIDpyb290ey0taW5rOiMyNTMyNGE7LS1zb2Z0OiM1YTZiODY7LS1saW5lOiNkZGU1ZjI7LS1pbjojZjY5MjFlOy0tb3V0OiMxNmEzYTM7LS1hYzojN2I1Y2Q2Oy0tYmc6I2VlZjNmYjt9CiAgKntib3gtc2l6aW5nOmJvcmRlci1ib3g7fWh0bWwsYm9keXttYXJnaW46MDtmb250LWZhbWlseToiU2Vnb2UgVUkiLHN5c3RlbS11aSxzYW5zLXNlcmlmO2NvbG9yOnZhcigtLWluayk7fQogIGJvZHl7YmFja2dyb3VuZDpyYWRpYWwtZ3JhZGllbnQoOTAwcHggNDAwcHggYXQgMzAlIC0xMCUsI2YzZjdmZix2YXIoLS1iZykpLHZhcigtLWJnKTtwYWRkaW5nOjE0cHg7fQogIGgxe2ZvbnQtc2l6ZToxN3B4O21hcmdpbjowIDAgMnB4O3RleHQtYWxpZ246Y2VudGVyO30KICAuc3Vie3RleHQtYWxpZ246Y2VudGVyO2NvbG9yOnZhcigtLXNvZnQpO2ZvbnQtc2l6ZToxMnB4O21hcmdpbjowIDAgMTBweDt9CiAgLmNhcmR7YmFja2dyb3VuZDojZmZmO2JvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSk7Ym9yZGVyLXJhZGl1czoxMnB4O3BhZGRpbmc6MTBweDtib3gtc2hhZG93OjAgNnB4IDE2cHggcmdiYSgxMjMsOTIsMjE0LC4wOCk7d2lkdGg6NTIwcHg7bWF4LXdpZHRoOjkydnc7bWFyZ2luOjAgYXV0bzt9CiAgY2FudmFze2Rpc3BsYXk6YmxvY2s7YmFja2dyb3VuZDojZmJmZGZmO2JvcmRlci1yYWRpdXM6OHB4O2JvcmRlcjoxcHggc29saWQgdmFyKC0tbGluZSk7d2lkdGg6MTAwJTtoZWlnaHQ6YXV0bzt9CiAgLnJlYWRvdXR7YmFja2dyb3VuZDojZjRmN2ZkO2JvcmRlci1yYWRpdXM6OHB4O3BhZGRpbmc6OHB4IDExcHg7Zm9udC1mYW1pbHk6bW9ub3NwYWNlO2ZvbnQtc2l6ZToxM3B4O21hcmdpbi10b3A6OHB4O2xpbmUtaGVpZ2h0OjEuODt9CiAgLnB7Y29sb3I6dmFyKC0taW4pO2ZvbnQtd2VpZ2h0OjcwMDt9LnR7Y29sb3I6dmFyKC0tYWMpO2ZvbnQtd2VpZ2h0OjcwMDt9Lmd7Y29sb3I6dmFyKC0tb3V0KTtmb250LXdlaWdodDo3MDA7fQogIC5yb3d7ZGlzcGxheTpmbGV4O2dhcDo4cHg7YWxpZ24taXRlbXM6Y2VudGVyO2p1c3RpZnktY29udGVudDpjZW50ZXI7bWFyZ2luOjEycHggMCAycHg7Zm9udC1zaXplOjEzcHg7ZmxleC13cmFwOndyYXA7fQogIGlucHV0W3R5cGU9cmFuZ2Vde3dpZHRoOjIzMHB4O21heC13aWR0aDo3MHZ3O2FjY2VudC1jb2xvcjp2YXIoLS1pbik7fQogIC5ub3Rle3RleHQtYWxpZ246Y2VudGVyO2ZvbnQtc2l6ZToxMS41cHg7Y29sb3I6dmFyKC0tc29mdCk7bWFyZ2luLXRvcDo4cHg7bWF4LXdpZHRoOjU2MHB4O21hcmdpbi1sZWZ0OmF1dG87bWFyZ2luLXJpZ2h0OmF1dG87bGluZS1oZWlnaHQ6MS41O30KPC9zdHlsZT48L2hlYWQ+PGJvZHk+CjxoMT5NU0UgbG9zcyBhbmQgaXRzIGdyYWRpZW50OiB0aGUgc2xvcGUgcG9pbnRzIGRvd25oaWxsPC9oMT4KPHAgY2xhc3M9InN1YiI+TG9zcyA9IChwcmVkaWN0aW9uIOKIkiB0YXJnZXQpwrIuIFRoZSBncmFkaWVudCA8Yj4yKHByZWQg4oiSIHRhcmdldCk8L2I+IGlzIHRoZSBzbG9wZSBvZiB0aGUgYm93bCDigJQgYmlnIHdoZW4geW91J3JlIGZhciBvZmYsIHplcm8gYXQgdGhlIGJvdHRvbSwgYW5kIGl0cyBzaWduIHNheXMgd2hpY2ggd2F5IHRvIG1vdmUuPC9wPgo8ZGl2IGNsYXNzPSJjYXJkIj48Y2FudmFzIGlkPSJjdiIgd2lkdGg9IjUyMCIgaGVpZ2h0PSIzMDAiPjwvY2FudmFzPjwvZGl2Pgo8ZGl2IGNsYXNzPSJyb3ciPnByZWRpY3Rpb24gPSA8aW5wdXQgaWQ9InAiIHR5cGU9InJhbmdlIiBtaW49Ii0zIiBtYXg9IjciIHN0ZXA9IjAuMSIgdmFsdWU9IjQuMiI+PHNwYW4gaWQ9InB2IiBzdHlsZT0iZm9udC1mYW1pbHk6bW9ub3NwYWNlO2NvbG9yOnZhcigtLWluKTtmb250LXdlaWdodDo3MDAiPjQuMjwvc3Bhbj4KJm5ic3A7wrcmbmJzcDsgdGFyZ2V0ID0gPGlucHV0IGlkPSJ0IiB0eXBlPSJyYW5nZSIgbWluPSItMyIgbWF4PSI3IiBzdGVwPSIwLjUiIHZhbHVlPSIyIj48c3BhbiBpZD0idHYiIHN0eWxlPSJmb250LWZhbWlseTptb25vc3BhY2U7Y29sb3I6dmFyKC0tYWMpO2ZvbnQtd2VpZ2h0OjcwMCI+Mi4wPC9zcGFuPjwvZGl2Pgo8ZGl2IGNsYXNzPSJyZWFkb3V0IiBzdHlsZT0ibWF4LXdpZHRoOjQ3MHB4O21hcmdpbi1sZWZ0OmF1dG87bWFyZ2luLXJpZ2h0OmF1dG87Ij4KICBsb3NzID0gKDxzcGFuIGNsYXNzPSJwIiBpZD0icjEiPjQuMjwvc3Bhbj4g4oiSIDxzcGFuIGNsYXNzPSJ0IiBpZD0icjIiPjIuMDwvc3Bhbj4pwrIgPSA8c3BhbiBpZD0icmwiPjQuODQ8L3NwYW4+PGJyPgogIGdyYWRpZW50ID0gMihwcmVkIOKIkiB0YXJnZXQpID0gMig8c3BhbiBjbGFzcz0icCI+wrc8L3NwYW4+KSA9IDxzcGFuIGNsYXNzPSJnIiBpZD0icmciPjQuNDA8L3NwYW4+ICZuYnNwOzxzcGFuIGlkPSJkaXIiIHN0eWxlPSJjb2xvcjp2YXIoLS1zb2Z0KSI+PC9zcGFuPgo8L2Rpdj4KPHAgY2xhc3M9Im5vdGUiPkluIGNvZGU6IDxjb2RlPm91dC5nID0gMi4qKG91dFs6LDBdLXRhcmcpWzosTm9uZV0vbjwvY29kZT4uIFRoYXQgZXh0cmEgPGNvZGU+L248L2NvZGU+IGp1c3QgYXZlcmFnZXMgdGhlIHBlci1zYW1wbGUgZ3JhZGllbnRzLiBUaGlzIHNpbmdsZSBudW1iZXIgaXMgd2hlcmUgdGhlIHdob2xlIGJhY2t3YXJkIHBhc3MgYmVnaW5zLjwvcD4KPHNjcmlwdD4KY29uc3QgJD1pZD0+ZG9jdW1lbnQuZ2V0RWxlbWVudEJ5SWQoaWQpOwpmdW5jdGlvbiByZW5kZXIoKXsKICBjb25zdCBwPSskKCdwJykudmFsdWUsdD0rJCgndCcpLnZhbHVlOwogICQoJ3B2JykudGV4dENvbnRlbnQ9cC50b0ZpeGVkKDEpOyQoJ3R2JykudGV4dENvbnRlbnQ9dC50b0ZpeGVkKDEpOwogICQoJ3IxJykudGV4dENvbnRlbnQ9cC50b0ZpeGVkKDEpOyQoJ3IyJykudGV4dENvbnRlbnQ9dC50b0ZpeGVkKDEpOwogIGNvbnN0IGxvc3M9KHAtdCkqKHAtdCksZz0yKihwLXQpOwogICQoJ3JsJykudGV4dENvbnRlbnQ9bG9zcy50b0ZpeGVkKDIpOyQoJ3JnJykudGV4dENvbnRlbnQ9Zy50b0ZpeGVkKDIpOwogICQoJ2RpcicpLnRleHRDb250ZW50ID0gTWF0aC5hYnMoZyk8MC4wNT8nKGF0IHRoZSBtaW5pbXVtIOKAlCBubyBwdXNoKSc6KGc+MD8nKHBvc2l0aXZlIOKGkiBsb3dlciB0aGUgcHJlZGljdGlvbiknOicobmVnYXRpdmUg4oaSIHJhaXNlIHRoZSBwcmVkaWN0aW9uKScpOwogIGNvbnN0IGN2PSQoJ2N2JyksY3R4PWN2LmdldENvbnRleHQoJzJkJyksVz1jdi53aWR0aCxIPWN2LmhlaWdodCxwYWQ9MzQ7CiAgY3R4LmNsZWFyUmVjdCgwLDAsVyxIKTsKICBjb25zdCB4bWluPS0zLHhtYXg9Nyx5bWluPTAseW1heD1NYXRoLm1heCgxNiwoKE1hdGgubWF4KE1hdGguYWJzKHhtYXgtdCksTWF0aC5hYnMoeG1pbi10KSkpKioyKSk7CiAgY29uc3QgWD12PT5wYWQrKHYteG1pbikvKHhtYXgteG1pbikqKFctMipwYWQpOwogIGNvbnN0IFk9dj0+SC1wYWQtKHYteW1pbikvKHltYXgteW1pbikqKEgtMipwYWQpOwogIGN0eC5zdHJva2VTdHlsZT0nI2UzZTlmNCc7Y3R4LmxpbmVXaWR0aD0xOwogIGN0eC5iZWdpblBhdGgoKTtjdHgubW92ZVRvKHBhZCxZKDApKTtjdHgubGluZVRvKFctcGFkLFkoMCkpO2N0eC5zdHJva2UoKTsKICAvLyB0YXJnZXQgbWFya2VyCiAgY3R4LnN0cm9rZVN0eWxlPScjN2I1Y2Q2JztjdHguc2V0TGluZURhc2goWzUsNF0pO2N0eC5saW5lV2lkdGg9MS41OwogIGN0eC5iZWdpblBhdGgoKTtjdHgubW92ZVRvKFgodCkscGFkKTtjdHgubGluZVRvKFgodCksSC1wYWQpO2N0eC5zdHJva2UoKTtjdHguc2V0TGluZURhc2goW10pOwogIGN0eC5maWxsU3R5bGU9JyM3YjVjZDYnO2N0eC5mb250PScxMXB4IG1vbm9zcGFjZSc7Y3R4LmZpbGxUZXh0KCd0YXJnZXQnLFgodCkrNCxwYWQrMTIpOwogIC8vIHBhcmFib2xhCiAgY3R4LnN0cm9rZVN0eWxlPScjZjY5MjFlJztjdHgubGluZVdpZHRoPTM7Y3R4LmJlZ2luUGF0aCgpO2xldCBzdD1mYWxzZTsKICBmb3IobGV0IHB4PXBhZDtweDw9Vy1wYWQ7cHgrKyl7Y29uc3QgeHY9eG1pbisocHgtcGFkKS8oVy0yKnBhZCkqKHhtYXgteG1pbik7Y29uc3QgeXY9KHh2LXQpKih4di10KTsKICAgIGlmKHl2PnltYXgpe3N0PWZhbHNlO2NvbnRpbnVlO31jb25zdCBweT1ZKHl2KTtpZighc3Qpe2N0eC5tb3ZlVG8ocHgscHkpO3N0PXRydWU7fWVsc2UgY3R4LmxpbmVUbyhweCxweSk7fQogIGN0eC5zdHJva2UoKTsKICAvLyBwb2ludCArIHRhbmdlbnQKICBjb25zdCBweT1ZKGxvc3MpO2N0eC5maWxsU3R5bGU9JyNmNjkyMWUnO2N0eC5iZWdpblBhdGgoKTtjdHguYXJjKFgocCkscHksNiwwLDcpO2N0eC5maWxsKCk7Y3R4LnN0cm9rZVN0eWxlPScjZmZmJztjdHgubGluZVdpZHRoPTI7Y3R4LnN0cm9rZSgpOwogIC8vIHRhbmdlbnQgbGluZSBzbG9wZSBnIGluIGRhdGEgY29vcmRzOyBjb252ZXJ0IHRvIHBpeGVscwogIGNvbnN0IHN4PShXLTIqcGFkKS8oeG1heC14bWluKSwgc3k9KEgtMipwYWQpLyh5bWF4LXltaW4pOwogIGNvbnN0IHNsb3BlUHg9LWcqc3kvc3g7IGNvbnN0IGxlbj03MDsKICBjdHguc3Ryb2tlU3R5bGU9JyMxNmEzYTMnO2N0eC5saW5lV2lkdGg9Mi41O2N0eC5iZWdpblBhdGgoKTsKICBjdHgubW92ZVRvKFgocCktbGVuLHB5LXNsb3BlUHgqKC1sZW4pKTtjdHgubGluZVRvKFgocCkrbGVuLHB5LXNsb3BlUHgqKGxlbikpO2N0eC5zdHJva2UoKTsKICBjdHguZmlsbFN0eWxlPScjMTZhM2EzJztjdHguZm9udD0nYm9sZCAxMXB4IG1vbm9zcGFjZSc7Y3R4LmZpbGxUZXh0KCdzbG9wZSA9ICcrZy50b0ZpeGVkKDEpLFgocCkrOCxweS0xMCk7CiAgY3R4LmZpbGxTdHlsZT0nIzlmYjBjYyc7Y3R4LmZvbnQ9JzEwcHggbW9ub3NwYWNlJztjdHguZmlsbFRleHQoJ3ByZWRpY3Rpb24nLFctcGFkLTYyLEgtcGFkKzE2KTsKfQokKCdwJykub25pbnB1dD1yZW5kZXI7JCgndCcpLm9uaW5wdXQ9cmVuZGVyO3JlbmRlcigpOwo8L3NjcmlwdD48L2JvZHk+PC9odG1sPgo=" style="width:100%; height:620px; border:1px solid #dde5f2; border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);" loading="lazy" title="MSE loss and its gradient"></iframe>')


In [ ]:
# ============================================================================
# COMPUTING THE INITIAL LOSS
# ============================================================================
# Before training, let's see how wrong our random network is!

initial_loss = mse(preds, y_train)

print(f"Initial MSE Loss: {initial_loss:.2f}")
print()
print("This is HIGH because our weights are random.")
print("After training, we want this number to be as LOW as possible.")
print()
print("Context:")
print("  - Perfect predictions would give loss = 0")
print("  - Our labels range from 0 to 9")
print("  - Average squared error of ~4309 means average error of ~66")
print("  - That's way off! (but expected with random weights)")

---
## Part 2: The Backward Pass - Computing Gradients

This is where the magic happens! The **backward pass** (backpropagation) computes how much each weight contributed to the error, so we know how to adjust them.

### The Chain Rule: Foundation of Backpropagation

The **chain rule** from calculus tells us how to compute derivatives of composed functions:

$$\frac{d}{dx}[f(g(x))] = f'(g(x)) \cdot g'(x)$$

In neural networks, we have many composed functions:
- `loss = MSE(Linear2(ReLU(Linear1(input))))`

To find how changing a weight affects the loss, we multiply derivatives along the path.

### Visual Understanding of Chain Rule

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        THE CHAIN RULE IN ACTION                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   INPUT ──► LINEAR1 ──► ReLU ──► LINEAR2 ──► LOSS                          │
│     x         l1        l2        out        L                              │
│                                                                             │
│   FORWARD: Data flows left to right                                         │
│   BACKWARD: Gradients flow right to left                                    │
│                                                                             │
│   ∂L/∂w1 = ∂L/∂out × ∂out/∂l2 × ∂l2/∂l1 × ∂l1/∂w1                         │
│            ─────────────────────────────────────────                        │
│            Chain rule: multiply all derivatives along the path              │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Key Insight: Storing Intermediate Values

During the forward pass, we **save intermediate results** (l1, l2, out) because we need them to compute gradients in the backward pass!

### 🎮 Interactive: the chain rule is just multiplication

Step **Forward →** to fill in values, then **Backward ←** to fill in gradients. Watch how each gradient is simply *the gradient coming back* times *one local derivative*. That repeated multiplication, walked from the loss back to the input, **is** backpropagation.

In [ ]:
# ============================================================================
# INTERACTIVE (embedded figure) -- run this cell. Self-contained iframe;
# works in Jupyter, Colab, and the exported HTML. Re-run to reset it.
# ============================================================================
from IPython.display import HTML
HTML('<iframe src="data:text/html;base64,PCFET0NUWVBFIGh0bWw+PGh0bWwgbGFuZz0iZW4iPjxoZWFkPjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+Q2hhaW4gcnVsZTwvdGl0bGU+CjxzdHlsZT4KICA6cm9vdHstLWluazojMjUzMjRhOy0tc29mdDojNWE2Yjg2Oy0tbGluZTojZGRlNWYyOy0taW46I2Y2OTIxZTstLW91dDojMTZhM2EzOy0tYWM6IzdiNWNkNjstLWJnOiNlZWYzZmI7fQogICp7Ym94LXNpemluZzpib3JkZXItYm94O30gaHRtbCxib2R5e21hcmdpbjowO2ZvbnQtZmFtaWx5OiJTZWdvZSBVSSIsc3lzdGVtLXVpLHNhbnMtc2VyaWY7Y29sb3I6dmFyKC0taW5rKTt9CiAgYm9keXtiYWNrZ3JvdW5kOnJhZGlhbC1ncmFkaWVudCg5MDBweCA0MDBweCBhdCAzMCUgLTEwJSwjZjNmN2ZmLHZhcigtLWJnKSksdmFyKC0tYmcpO3BhZGRpbmc6MTRweDt9CiAgaDF7Zm9udC1zaXplOjE3cHg7bWFyZ2luOjAgMCAycHg7dGV4dC1hbGlnbjpjZW50ZXI7fQogIC5zdWJ7dGV4dC1hbGlnbjpjZW50ZXI7Y29sb3I6dmFyKC0tc29mdCk7Zm9udC1zaXplOjEycHg7bWFyZ2luOjAgMCAxMnB4O30KICAuZ3JhcGh7ZGlzcGxheTpmbGV4O2FsaWduLWl0ZW1zOmNlbnRlcjtqdXN0aWZ5LWNvbnRlbnQ6Y2VudGVyO2dhcDowO2ZsZXgtd3JhcDp3cmFwO21hcmdpbjo2cHggMCAxMHB4O30KICAubm9kZXtiYWNrZ3JvdW5kOiNmZmY7Ym9yZGVyOjJweCBzb2xpZCB2YXIoLS1saW5lKTtib3JkZXItcmFkaXVzOjEycHg7cGFkZGluZzo4cHggMTBweDttaW4td2lkdGg6NzhweDt0ZXh0LWFsaWduOmNlbnRlcjtib3gtc2hhZG93OjAgNnB4IDE2cHggcmdiYSgxMjMsOTIsMjE0LC4xMCk7fQogIC5ub2RlIC5uYW1le2ZvbnQtd2VpZ2h0OjcwMDtmb250LXNpemU6MTJweDt9CiAgLm5vZGUgLnZhbHtmb250LWZhbWlseToiU0YgTW9ubyIsQ29uc29sYXMsbW9ub3NwYWNlO2ZvbnQtc2l6ZToxMnB4O2NvbG9yOnZhcigtLWluKTttYXJnaW4tdG9wOjJweDt9CiAgLm5vZGUgLmdyYWR7Zm9udC1mYW1pbHk6IlNGIE1vbm8iLENvbnNvbGFzLG1vbm9zcGFjZTtmb250LXNpemU6MTFweDtjb2xvcjp2YXIoLS1vdXQpO21hcmdpbi10b3A6MnB4O21pbi1oZWlnaHQ6MTRweDt9CiAgLmFycm93e3Bvc2l0aW9uOnJlbGF0aXZlO3dpZHRoOjQ2cHg7aGVpZ2h0OjJweDtiYWNrZ3JvdW5kOnZhcigtLWxpbmUpO21hcmdpbjowIDJweDthbGlnbi1zZWxmOmZsZXgtc3RhcnQ7bWFyZ2luLXRvcDozNHB4O30KICAuYXJyb3cgLmxhYntwb3NpdGlvbjphYnNvbHV0ZTt0b3A6LTMwcHg7bGVmdDo1MCU7dHJhbnNmb3JtOnRyYW5zbGF0ZVgoLTUwJSk7Zm9udC1mYW1pbHk6bW9ub3NwYWNlO2ZvbnQtc2l6ZToxMHB4O2NvbG9yOnZhcigtLWFjKTt3aGl0ZS1zcGFjZTpub3dyYXA7YmFja2dyb3VuZDojZmZmO3BhZGRpbmc6MCAzcHg7Ym9yZGVyLXJhZGl1czo0cHg7fQogIC5hcnJvdyAuZ2xhYntwb3NpdGlvbjphYnNvbHV0ZTt0b3A6OHB4O2xlZnQ6NTAlO3RyYW5zZm9ybTp0cmFuc2xhdGVYKC01MCUpO2ZvbnQtZmFtaWx5Om1vbm9zcGFjZTtmb250LXNpemU6MTBweDtjb2xvcjp2YXIoLS1vdXQpO3doaXRlLXNwYWNlOm5vd3JhcDttaW4taGVpZ2h0OjEycHg7fQogIC5hcnJvdy5hY3RpdmV7YmFja2dyb3VuZDp2YXIoLS1hYyk7aGVpZ2h0OjNweDt9CiAgLm5vZGUuYWN0aXZle2JvcmRlci1jb2xvcjp2YXIoLS1hYyk7Ym94LXNoYWRvdzowIDAgMCAzcHggcmdiYSgxMjMsOTIsMjE0LC4xOCk7fQogIC5wYW5lbHtiYWNrZ3JvdW5kOiNmZmY7Ym9yZGVyOjFweCBzb2xpZCB2YXIoLS1saW5lKTtib3JkZXItcmFkaXVzOjEycHg7cGFkZGluZzoxMHB4IDEzcHg7Zm9udC1zaXplOjEzcHg7bGluZS1oZWlnaHQ6MS41O21pbi1oZWlnaHQ6NjRweDt9CiAgLnBhbmVsIGJ7Y29sb3I6dmFyKC0tYWMpO30KICAuZXF7Zm9udC1mYW1pbHk6bW9ub3NwYWNlO2JhY2tncm91bmQ6I2Y0ZjdmZDtib3JkZXItcmFkaXVzOjhweDtwYWRkaW5nOjZweCA5cHg7bWFyZ2luLXRvcDo2cHg7Zm9udC1zaXplOjEycHg7Y29sb3I6dmFyKC0taW5rKTtvdmVyZmxvdy14OmF1dG87fQogIC5jdHJ7ZGlzcGxheTpmbGV4O2dhcDo4cHg7anVzdGlmeS1jb250ZW50OmNlbnRlcjttYXJnaW46MTJweCAwIDhweDtmbGV4LXdyYXA6d3JhcDt9CiAgYnV0dG9ue2ZvbnQtZmFtaWx5OmluaGVyaXQ7Zm9udC13ZWlnaHQ6NzAwO2ZvbnQtc2l6ZToxM3B4O2JvcmRlcjpub25lO2JvcmRlci1yYWRpdXM6OXB4O3BhZGRpbmc6OHB4IDE1cHg7Y3Vyc29yOnBvaW50ZXI7fQogIC5uZXh0e2JhY2tncm91bmQ6dmFyKC0tYWMpO2NvbG9yOiNmZmY7fSAucmVzZXR7YmFja2dyb3VuZDojZWNlN2ZiO2NvbG9yOnZhcigtLWFjKTt9CiAgLnJvd3tkaXNwbGF5OmZsZXg7Z2FwOjEwcHg7YWxpZ24taXRlbXM6Y2VudGVyO2p1c3RpZnktY29udGVudDpjZW50ZXI7Zm9udC1zaXplOjEycHg7Y29sb3I6dmFyKC0tc29mdCk7bWFyZ2luLXRvcDo0cHg7fQogIGlucHV0W3R5cGU9cmFuZ2Vde2FjY2VudC1jb2xvcjp2YXIoLS1pbik7fQogIC5sZWdlbmR7dGV4dC1hbGlnbjpjZW50ZXI7Zm9udC1zaXplOjExcHg7Y29sb3I6dmFyKC0tc29mdCk7bWFyZ2luLXRvcDo2cHg7fQogIC5sZWdlbmQgLmF7Y29sb3I6dmFyKC0taW4pO2ZvbnQtd2VpZ2h0OjcwMDt9LmxlZ2VuZCAuYntjb2xvcjp2YXIoLS1vdXQpO2ZvbnQtd2VpZ2h0OjcwMDt9Cjwvc3R5bGU+PC9oZWFkPjxib2R5Pgo8aDE+VGhlIGNoYWluIHJ1bGUsIG1hZGUgb2YgbXVsdGlwbGljYXRpb25zPC9oMT4KPHAgY2xhc3M9InN1YiI+QSB0aW55IG5ldHdvcmsgPGI+TCA9IChhwrd4KcKyPC9iPi4gRm9yd2FyZCBmaWxscyA8c3BhbiBzdHlsZT0iY29sb3I6I2Y2OTIxZTtmb250LXdlaWdodDo3MDAiPnZhbHVlczwvc3Bhbj47IGJhY2t3YXJkIGZpbGxzIDxzcGFuIHN0eWxlPSJjb2xvcjojMTZhM2EzO2ZvbnQtd2VpZ2h0OjcwMCI+Z3JhZGllbnRzPC9zcGFuPiDigJQgZWFjaCBpcyBqdXN0IHRoZSBuZXh0IGdyYWRpZW50IMOXIG9uZSBsb2NhbCBkZXJpdmF0aXZlLjwvcD4KCjxkaXYgY2xhc3M9ImdyYXBoIj4KICA8ZGl2IGNsYXNzPSJub2RlIiBpZD0ibngiPjxkaXYgY2xhc3M9Im5hbWUiPng8L2Rpdj48ZGl2IGNsYXNzPSJ2YWwiIGlkPSJ2eCI+Mi4wMDwvZGl2PjxkaXYgY2xhc3M9ImdyYWQiIGlkPSJneCI+PC9kaXY+PC9kaXY+CiAgPGRpdiBjbGFzcz0iYXJyb3ciIGlkPSJhMSI+PGRpdiBjbGFzcz0ibGFiIiBpZD0ibDEiPsOXIGE8L2Rpdj48ZGl2IGNsYXNzPSJnbGFiIiBpZD0iZzEiPjwvZGl2PjwvZGl2PgogIDxkaXYgY2xhc3M9Im5vZGUiIGlkPSJudSI+PGRpdiBjbGFzcz0ibmFtZSI+dSA9IGHCt3g8L2Rpdj48ZGl2IGNsYXNzPSJ2YWwiIGlkPSJ2dSI+PC9kaXY+PGRpdiBjbGFzcz0iZ3JhZCIgaWQ9Imd1Ij48L2Rpdj48L2Rpdj4KICA8ZGl2IGNsYXNzPSJhcnJvdyIgaWQ9ImEyIj48ZGl2IGNsYXNzPSJsYWIiPijCtynCsjwvZGl2PjxkaXYgY2xhc3M9ImdsYWIiIGlkPSJnMiI+PC9kaXY+PC9kaXY+CiAgPGRpdiBjbGFzcz0ibm9kZSIgaWQ9Im5MIj48ZGl2IGNsYXNzPSJuYW1lIj5MID0gdcKyPC9kaXY+PGRpdiBjbGFzcz0idmFsIiBpZD0idkwiPjwvZGl2PjxkaXYgY2xhc3M9ImdyYWQiIGlkPSJnTCI+PC9kaXY+PC9kaXY+CjwvZGl2PgoKPGRpdiBjbGFzcz0icGFuZWwiIGlkPSJwYW5lbCI+UHJlc3MgPGI+Rm9yd2FyZCDihpI8L2I+IHRvIHB1c2ggdmFsdWVzIGxlZnQtdG8tcmlnaHQsIHRoZW4gPGI+QmFja3dhcmQg4oaQPC9iPiB0byBwdWxsIGdyYWRpZW50cyByaWdodC10by1sZWZ0LjwvZGl2PgoKPGRpdiBjbGFzcz0icm93Ij5hID0gPGlucHV0IHR5cGU9InJhbmdlIiBpZD0iYSIgbWluPSIwLjUiIG1heD0iMyIgc3RlcD0iMC41IiB2YWx1ZT0iMS41Ij48c3BhbiBpZD0iYXYiIHN0eWxlPSJmb250LWZhbWlseTptb25vc3BhY2U7Y29sb3I6IzdiNWNkNjtmb250LXdlaWdodDo3MDAiPjEuNTwvc3Bhbj4gJm5ic3A7wrcmbmJzcDsgeCA9IDxpbnB1dCB0eXBlPSJyYW5nZSIgaWQ9IngiIG1pbj0iLTMiIG1heD0iMyIgc3RlcD0iMC41IiB2YWx1ZT0iMiI+PHNwYW4gaWQ9Inh2IiBzdHlsZT0iZm9udC1mYW1pbHk6bW9ub3NwYWNlO2NvbG9yOiNmNjkyMWU7Zm9udC13ZWlnaHQ6NzAwIj4yLjA8L3NwYW4+PC9kaXY+CjxkaXYgY2xhc3M9ImN0ciI+CiAgPGJ1dHRvbiBjbGFzcz0ibmV4dCIgaWQ9InN0ZXAiPkZvcndhcmQg4oaSPC9idXR0b24+CiAgPGJ1dHRvbiBjbGFzcz0icmVzZXQiIGlkPSJyZXNldCI+4oa6IFJlc2V0PC9idXR0b24+CjwvZGl2Pgo8ZGl2IGNsYXNzPSJsZWdlbmQiPjxzcGFuIGNsYXNzPSJhIj5vcmFuZ2UgPSBmb3J3YXJkIHZhbHVlPC9zcGFuPiAmbmJzcDvCtyZuYnNwOyA8c3BhbiBjbGFzcz0iYiI+dGVhbCA9IOKIgkwv4oiCKHRoYXQgbm9kZSk8L3NwYW4+PC9kaXY+Cgo8c2NyaXB0Pgpjb25zdCAkPWlkPT5kb2N1bWVudC5nZXRFbGVtZW50QnlJZChpZCk7CmxldCBhPTEuNSx4PTIscGhhc2U9MDsgLy8gMCByZWFkeSwgMS4uMiBmb3J3YXJkLCAzLi41IGJhY2t3YXJkLCA2IGRvbmUKZnVuY3Rpb24gdmFscygpe2NvbnN0IHU9YSp4LEw9dSp1O3JldHVybnt1LEwsCiAgZExfZEw6MSwgZExfZHU6Mip1LCBkTF9keDoyKnUqYX07fQpmdW5jdGlvbiBmKG4pe3JldHVybiBuLnRvRml4ZWQoMik7fQpmdW5jdGlvbiByZW5kZXIoKXsKICAkKCdhdicpLnRleHRDb250ZW50PWEudG9GaXhlZCgxKTskKCd4dicpLnRleHRDb250ZW50PXgudG9GaXhlZCgxKTsKICBjb25zdCB2PXZhbHMoKTskKCd2eCcpLnRleHRDb250ZW50PWYoeCk7CiAgWyAnbngnLCdudScsJ25MJywnYTEnLCdhMiddLmZvckVhY2goaT0+JChpKS5jbGFzc0xpc3QucmVtb3ZlKCdhY3RpdmUnKSk7CiAgLy8gZm9yd2FyZCBmaWxscwogICQoJ3Z1JykudGV4dENvbnRlbnQgPSBwaGFzZT49MT8gZih2LnUpOicnOwogICQoJ3ZMJykudGV4dENvbnRlbnQgPSBwaGFzZT49Mj8gZih2LkwpOicnOwogIC8vIGJhY2t3YXJkIGZpbGxzCiAgJCgnZ0wnKS50ZXh0Q29udGVudCA9IHBoYXNlPj0zPyAn4oiCTC/iiIJMPTEnOicnOwogICQoJ2cyJykudGV4dENvbnRlbnQgPSBwaGFzZT49ND8gJ8OXMnU9JytmKHYuZExfZHUpOicnOwogICQoJ2d1JykudGV4dENvbnRlbnQgPSBwaGFzZT49ND8gZih2LmRMX2R1KTonJzsKICAkKCdnMScpLnRleHRDb250ZW50ID0gcGhhc2U+PTU/ICfDl2E9JytmKGEpOicnOwogICQoJ2d4JykudGV4dENvbnRlbnQgPSBwaGFzZT49NT8gZih2LmRMX2R4KTonJzsKICBsZXQgaHRtbD0nJyxidG49J0ZvcndhcmQg4oaSJzsKICBpZihwaGFzZT09PTApe2h0bWw9J1ByZXNzIDxiPkZvcndhcmQg4oaSPC9iPiB0byBjb21wdXRlIHZhbHVlcyBsZWZ0LXRvLXJpZ2h0Lic7fQogIGlmKHBoYXNlPT09MSl7JCgnbngnKS5jbGFzc0xpc3QuYWRkKCdhY3RpdmUnKTskKCdhMScpLmNsYXNzTGlzdC5hZGQoJ2FjdGl2ZScpOyQoJ251JykuY2xhc3NMaXN0LmFkZCgnYWN0aXZlJyk7CiAgICBodG1sPSc8Yj5Gb3J3YXJkLCBzdGVwIDEuPC9iPiBNdWx0aXBseTogdSA9IGHCt3ggPSAnK2EudG9GaXhlZCgxKSsnwrcnK2YoeCkrJyA9IDxiPicrZih2LnUpKyc8L2I+Lic7fQogIGlmKHBoYXNlPT09Mil7JCgnbnUnKS5jbGFzc0xpc3QuYWRkKCdhY3RpdmUnKTskKCdhMicpLmNsYXNzTGlzdC5hZGQoJ2FjdGl2ZScpOyQoJ25MJykuY2xhc3NMaXN0LmFkZCgnYWN0aXZlJyk7CiAgICBodG1sPSc8Yj5Gb3J3YXJkLCBzdGVwIDIuPC9iPiBTcXVhcmU6IEwgPSB1wrIgPSAnK2Yodi51KSsnwrIgPSA8Yj4nK2Yodi5MKSsnPC9iPi4gRm9yd2FyZCBkb25lIOKAlCBub3cgcHJlc3MgPGI+QmFja3dhcmQg4oaQPC9iPi4nO2J0bj0nQmFja3dhcmQg4oaQJzt9CiAgaWYocGhhc2U9PT0zKXskKCduTCcpLmNsYXNzTGlzdC5hZGQoJ2FjdGl2ZScpOwogICAgaHRtbD0nPGI+QmFja3dhcmQgc3RhcnRzIGF0IHRoZSBlbmQuPC9iPiDiiIJML+KIgkwgPSAxICh0aGUgcmF0ZSBvZiBjaGFuZ2Ugb2YgYSB2YWx1ZSB3LnIudC4gaXRzZWxmKS48ZGl2IGNsYXNzPSJlcSI+Z3JhZCBzbyBmYXIgPSAxPC9kaXY+JztidG49J0JhY2t3YXJkIOKGkCc7fQogIGlmKHBoYXNlPT09NCl7JCgnYTInKS5jbGFzc0xpc3QuYWRkKCdhY3RpdmUnKTskKCdudScpLmNsYXNzTGlzdC5hZGQoJ2FjdGl2ZScpOwogICAgaHRtbD0nPGI+VGhyb3VnaCB0aGUgc3F1YXJlLjwvYj4gTG9jYWwgZGVyaXZhdGl2ZSBvZiB1wrIgaXMgMnUuIE11bHRpcGx5IGl0IG9udG8gdGhlIGdyYWRpZW50IGNvbWluZyBiYWNrOjxkaXYgY2xhc3M9ImVxIj7iiIJML+KIgnUgPSAo4oiCTC/iiIJMKSDCtyAydSA9IDEgwrcgJytmKHYuZExfZHUpKycgPSA8Yj4nK2Yodi5kTF9kdSkrJzwvYj48L2Rpdj4nO2J0bj0nQmFja3dhcmQg4oaQJzt9CiAgaWYocGhhc2U9PT01KXskKCdhMScpLmNsYXNzTGlzdC5hZGQoJ2FjdGl2ZScpOyQoJ254JykuY2xhc3NMaXN0LmFkZCgnYWN0aXZlJyk7CiAgICBodG1sPSc8Yj5UaHJvdWdoIHRoZSBtdWx0aXBseS48L2I+IExvY2FsIGRlcml2YXRpdmUgb2YgYcK3eCB3LnIudC4geCBpcyBhLiBNdWx0aXBseSBhZ2Fpbjo8ZGl2IGNsYXNzPSJlcSI+4oiCTC/iiIJ4ID0gKOKIgkwv4oiCdSkgwrcgYSA9ICcrZih2LmRMX2R1KSsnIMK3ICcrYS50b0ZpeGVkKDEpKycgPSA8Yj4nK2Yodi5kTF9keCkrJzwvYj48L2Rpdj5UaGF0IGNoYWluIOKAlCA8Yj4xIMK3IDJ1IMK3IGE8L2I+IOKAlCBpcyB0aGUgY2hhaW4gcnVsZTogd2FsayBiYWNrLCBtdWx0aXBseSBvbmUgbG9jYWwgZGVyaXZhdGl2ZSBhdCBlYWNoIHN0ZXAuJztidG49J0RvbmUnO30KICBpZihwaGFzZT49NSl7YnRuPSdEb25lJzt9CiAgJCgncGFuZWwnKS5pbm5lckhUTUw9aHRtbDskKCdzdGVwJykudGV4dENvbnRlbnQ9cGhhc2U+PTU/J0RvbmUnOmJ0bjskKCdzdGVwJykuZGlzYWJsZWQ9cGhhc2U+PTU7Cn0KJCgnc3RlcCcpLm9uY2xpY2s9KCk9PntpZihwaGFzZTw1KXtwaGFzZSsrO3JlbmRlcigpO319OwokKCdyZXNldCcpLm9uY2xpY2s9KCk9PntwaGFzZT0wO3JlbmRlcigpO307CiQoJ2EnKS5vbmlucHV0PWU9PnthPStlLnRhcmdldC52YWx1ZTtwaGFzZT0wO3JlbmRlcigpO307CiQoJ3gnKS5vbmlucHV0PWU9Pnt4PStlLnRhcmdldC52YWx1ZTtwaGFzZT0wO3JlbmRlcigpO307CnJlbmRlcigpOwo8L3NjcmlwdD48L2JvZHk+PC9odG1sPgo=" style="width:100%; height:360px; border:1px solid #dde5f2; border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);" loading="lazy" title="The chain rule"></iframe>')


# Deep Dive: the chain rule, and why backprop is one backward sweep

---

## One operation at a time

Calculus gives us the **chain rule** for a composition of two functions:

$$\frac{d}{dx}\,f\big(g(x)\big) \;=\; f'\big(g(x)\big)\,\cdot\, g'(x).$$

Read it as a recipe: *the derivative of the outside (evaluated at the inside) times the derivative of the inside.* Our network is a deep composition,

$$L \;=\; \text{mse}\Big(\text{lin}_2\big(\text{relu}(\text{lin}_1(x))\big),\, y\Big),$$

so to get `∂L/∂w₁` we multiply the local derivative of every operation along the path from `w₁` to `L`:

$$\frac{\partial L}{\partial w_1} \;=\; \frac{\partial L}{\partial \text{out}}\cdot\frac{\partial \text{out}}{\partial l_2}\cdot\frac{\partial l_2}{\partial l_1}\cdot\frac{\partial l_1}{\partial w_1}.$$

## The key efficiency idea

Notice the **front part of that product is shared** by every parameter. `∂L/∂out` is needed for `w₂`, `b₂`, *and* (multiplied by more terms) for `w₁`, `b₁`. So instead of recomputing the whole chain for each weight, we compute it **once, right-to-left**, carrying a running "gradient so far" and handing it to the next layer back. Each layer receives `∂L/∂(its output)` and produces `∂L/∂(its input)` to pass on, plus `∂L/∂(its own weights)` to keep.

That running quantity — the gradient of the loss with respect to a layer's output — is exactly what the code stores as `out.g`. The whole backward pass is: **seed `out.g` at the loss, then each layer multiplies by its local derivative and passes the result back.**

## Why we save the forward values

Look again at `f'(g(x))` — the outer derivative is evaluated *at the inner value*. So to compute gradients we need the intermediate activations from the forward pass (`l1`, `l2`, `out`). That is why the forward pass **saves** them: the backward pass reuses them. Memory for speed — this trade-off is the reason training a network uses far more memory than just running it.

In [ ]:
# ============================================================================
# SYMBOLIC DIFFERENTIATION WITH SYMPY
# ============================================================================
# Before diving into backprop, let's see how derivatives work using SymPy,
# a library for symbolic mathematics. This helps us verify our gradient formulas.

from sympy import symbols, diff

# Create symbolic variables x and y
# These are not numbers - they represent mathematical variables
x, y = symbols('x y')

# diff(expression, variable) computes the derivative
# d/dx (x²) = 2x
print("Derivative of x² with respect to x:")
print(f"  d/dx(x²) = {diff(x**2, x)}")
print()
print("Why? Using the power rule: d/dx(xⁿ) = n×xⁿ⁻¹")
print("      d/dx(x²) = 2×x¹ = 2x")

In [ ]:
# ============================================================================
# ANOTHER DERIVATIVE EXAMPLE
# ============================================================================
# d/dx (3x² + 9) = 6x
# 
# Breaking it down:
#   - d/dx(3x²) = 3 × d/dx(x²) = 3 × 2x = 6x  (constant multiple rule + power rule)
#   - d/dx(9) = 0  (derivative of a constant is zero)
#   - Total: 6x + 0 = 6x

result = diff(3*x**2 + 9, x)
print(f"d/dx(3x² + 9) = {result}")
print()
print("This is important for MSE! Our loss is: (prediction - target)²")
print("The derivative of (prediction - target)² with respect to prediction is:")
print("  2 × (prediction - target)")
print()
print("This tells us: the gradient is proportional to the error!")

In [ ]:
# ============================================================================
# GRADIENT OF A LINEAR LAYER
# ============================================================================
# This function computes gradients for a linear layer: output = input @ weight + bias
#
# INPUTS:
#   inp  - The input to this layer (saved during forward pass)
#   out  - The output of this layer (saved during forward pass)
#   w    - The weight matrix
#   b    - The bias vector
#
# ASSUMPTION: out.g already contains ∂Loss/∂output (the gradient flowing back)
#
# OUTPUT: Sets inp.g, w.g, and b.g (gradients stored as attributes)

def lin_grad(inp, out, w, b):
    """
    Compute gradients for a linear layer.
    
    This implements the backward pass for: output = input @ weight + bias
    
    Args:
        inp: Input tensor that was fed to this layer during forward pass
             Shape: (batch_size, input_features)
             Example: (50000, 784) for our training data to layer 1
        
        out: Output tensor that this layer produced during forward pass
             Shape: (batch_size, output_features)
             Example: (50000, 50) for layer 1 output
             IMPORTANT: out.g must already be set (gradient from next layer)
        
        w:   Weight matrix for this layer
             Shape: (input_features, output_features)
             Example: (784, 50) for layer 1
        
        b:   Bias vector for this layer
             Shape: (output_features,)
             Example: (50,) for layer 1
    
    Sets:
        inp.g: Gradient with respect to input (∂Loss/∂input)
        w.g:   Gradient with respect to weights (∂Loss/∂weight)
        b.g:   Gradient with respect to bias (∂Loss/∂bias)
    """
    
    # ─────────────────────────────────────────────────────────────────────────
    # GRADIENT WITH RESPECT TO INPUT (inp.g)
    # ─────────────────────────────────────────────────────────────────────────
    # Formula: inp.g = out.g @ w.T
    #
    # Intuition: To find how input affects loss, we multiply the output gradient
    # by the transpose of weights. This "reverses" the forward computation.
    #
    # Shapes: out.g is (batch, out_features), w.T is (out_features, in_features)
    #         Result: (batch, in_features) - same shape as inp!
    #
    # w.t() is the transpose of w (swaps rows and columns)
    inp.g = out.g @ w.t()
    
    # ─────────────────────────────────────────────────────────────────────────
    # GRADIENT WITH RESPECT TO WEIGHTS (w.g)
    # ─────────────────────────────────────────────────────────────────────────
    # Formula: w.g = inp.T @ out.g  (simplified version, see below for original)
    #
    # The original code uses a more explicit form:
    #   (inp.unsqueeze(-1) * out.g.unsqueeze(1)).sum(0)
    #
    # Breaking this down:
    #   inp.unsqueeze(-1): Add dimension at end. (batch, in) → (batch, in, 1)
    #   out.g.unsqueeze(1): Add dimension at pos 1. (batch, out) → (batch, 1, out)
    #   Multiply: Broadcasting gives (batch, in, out)
    #   .sum(0): Sum across batch dimension → (in, out), same shape as w
    #
    # This computes: for each weight w[i,j], how much did it contribute to error?
    w.g = (inp.unsqueeze(-1) * out.g.unsqueeze(1)).sum(0)
    # w.g = inp.T @ out.g
    # ─────────────────────────────────────────────────────────────────────────
    # GRADIENT WITH RESPECT TO BIAS (b.g)
    # ─────────────────────────────────────────────────────────────────────────
    # Derivation:
    #   Forward Pass:  out[i,j] = Σ_k (inp[i,k] * w[k,j]) + b[j]
    #   Local Grad:    ∂out[i,j] / ∂b[j] = 1
    #   Chain Rule:    ∂Loss / ∂b[j] = Σ_i (∂Loss / ∂out[i,j] * ∂out[i,j] / ∂b[j])
    #                                = Σ_i (out.g[i,j] * 1)
    #
    # Why sum(0)?
    #   In the forward pass, the same bias vector 'b' is broadcasted and added 
    #   to every single sample in the batch. Because 'b' contributes to the 
    #   output of every sample, its total gradient is the sum of the gradients 
    #   from every sample in that batch.
    #
    # Shapes: out.g is (batch, out_features)
    #         sum(0) collapses the batch dimension → (out_features,)
    b.g = out.g.sum(0)

print("lin_grad function defined!")
print("This computes gradients for: output = input @ weight + bias")

**What does the code above do?**

Implements the **backward pass of one linear layer**. Given `out.g = ∂L/∂out` (handed in from the layer after), it fills three gradients:

| Line | Computes | Shape |
|---|---|---|
| `inp.g = out.g @ w.t()` | `∂L/∂input` — passed further back | (bs, in) |
| `w.g = (inp.unsqueeze(-1) * out.g.unsqueeze(1)).sum(0)` | `∂L/∂w` (≡ `inp.T @ out.g`) | (in, out) |
| `b.g = out.g.sum(0)` | `∂L/∂b` | (out,) |

The derivations for all three are in the deep dive below.

## Primer 1 — the chain rule *in partials*, and two matrix-calculus facts

Before the element-by-element derivations below, here are the few ideas that make every gradient line predictable. (Everything here is for **this exact network** — one hidden layer, ReLU, MSE — nothing more.)

**Write each gradient as (incoming) × (local).** For anything the loss reaches *through* a later quantity, the chain rule is
$$\frac{\partial L}{\partial(\text{thing})}=\underbrace{\frac{\partial L}{\partial(\text{next thing})}}_{\text{gradient already arriving}}\cdot\underbrace{\frac{\partial(\text{next thing})}{\partial(\text{thing})}}_{\text{local derivative}}.$$

For our two linear layers (with $G \equiv \partial L/\partial \hat Y$, i.e. `out.g`) that is, *before plugging in any values*:

| we want | = incoming × local | = matrix form | code line |
|---|---|---|---|
| $\partial L/\partial A_1$ | $\dfrac{\partial L}{\partial\hat Y}\cdot\dfrac{\partial\hat Y}{\partial A_1}$,&nbsp; local $=W_2$ | $G\,W_2^{\top}$ | `l2.g = out.g @ w2.t()` |
| $\partial L/\partial W_2$ | $\dfrac{\partial L}{\partial\hat Y}\cdot\dfrac{\partial\hat Y}{\partial W_2}$,&nbsp; local $=A_1$ | $A_1^{\top}G$ | `w2.g = l2.T @ out.g` |
| $\partial L/\partial b_2$ | $\dfrac{\partial L}{\partial\hat Y}\cdot\dfrac{\partial\hat Y}{\partial b_2}$,&nbsp; local $=1$ | $\sum_i G_{i\cdot}$ | `b2.g = out.g.sum(0)` |
| $\partial L/\partial Z_1$ | $\dfrac{\partial L}{\partial A_1}\cdot\dfrac{\partial A_1}{\partial Z_1}$,&nbsp; local $=\mathbb 1[Z_1>0]$ | $\dfrac{\partial L}{\partial A_1}\odot\mathbb 1[Z_1>0]$ | `l1.g = (l1>0).float()*l2.g` |

(Recall `l1` is $Z_1$, the pre-activation, and `l2` is $A_1 = \mathrm{ReLU}(Z_1)$. Layer 1 just repeats the layer-2 rows with $X$ in place of $A_1$ and `l1.g` in place of $G$.) The proofs below show **why** each local part has that value, and where the sum / transpose comes from.

**Fact A — a transpose just swaps row ↔ column.** $(M^{\top})_{rc}=M_{cr}$; no number changes, the same entries are relabelled. We reach for it only to line an index up for matrix multiply.

**Fact B — matrix multiply sums over the shared *inner* index.** By definition $(MN)_{rc}=\sum_t M_{rt}N_{tc}$ — the summed index $t$ is the **column of $M$** and the **row of $N$**. So whenever a gradient comes out as *"a sum over some index,"* we can write it as a matrix product, and the transpose is simply what slides that summed index into the inner slot. (Example: $\sum_i A_{1,ik}G_{ij}$ has $i$ as a *row* of $A_1$; transpose $A_1$ so $i$ becomes its column, and the sum is $(A_1^{\top}G)_{kj}$.)

**Shapes are a free check.** Every gradient has the **same shape as the thing it differentiates**, so `w2.g` must match `w2`, `inp.g` must match `inp`, etc. If a candidate formula gives the wrong shape, it is wrong — no calculus needed.

## Primer 2 — fan-out decides *which* sum appears (with concrete numbers)

One question fixes every sum and every transpose:

> **In the forward pass, what was this quantity copied or reused across? Backward, you _sum_ the gradient over exactly that.**

| forward (how it was reused) | backward (sum over…) | shows up as |
|---|---|---|
| a **weight** is reused for **every sample** | samples ($\sum_i$) | the `.T` in `inp.T @ out.g` |
| an **activation** feeds **every output unit** | outputs ($\sum_j$) | the `.T` in `out.g @ w.T` |
| a **bias** is **broadcast** over rows | rows ($\sum_i$) | `out.g.sum(0)` |
| ReLU sends one input to **one** output | nothing — no sum | `(l1>0)*l2.g` |

**Wire picture + a number (layer 2).** Think of the weights as **wires**: hidden unit $k$ connects to output $j$ with strength $W_{2,kj}$. Forward, a hidden unit *sends* its value down those wires; backward, it *collects blame* back from every output it fed, along the same wires. To make the output index do something, suppose for a moment the layer had **two** outputs (our real net has one, so its $\sum_j$ is just a single term): with
$$W_2=\begin{bmatrix}10&30\\20&40\end{bmatrix}\quad\text{and output error}\quad G=[\,0.5,\;0.1\,],$$
hidden unit 0's blame is $0.5\cdot10 + 0.1\cdot30 = 8$ — exactly $(G\,W_2^{\top})$. **Forward multiply by $W_2$; backward by $W_2^{\top}$** — the same wires, opposite direction. (And a *weight's* gradient sums over samples, while an *activation's* sums over outputs — the two transposes are mirror images.)

**Gate picture + a number (ReLU).** ReLU is a **gate**: forward it passes a positive number through unchanged (slope $1$) and blocks a negative one to $0$ (slope $0$); backward the gradient does the same. If three units have $Z_1=[\,2,\,-3,\,0.5\,]$, the mask `(l1>0)` is $[\,1,0,1\,]$, so an incoming $\partial L/\partial A_1=[\,0.4,\,0.6,\,-0.2\,]$ becomes
$$\frac{\partial L}{\partial Z_1}=[\,0.4,\,0.6,\,-0.2\,]\odot[\,1,0,1\,]=[\,0.4,\;\mathbf 0,\;-0.2\,].$$
The off unit gets **zero** blame — nudging $Z_1=-3$ to $-3.01$ leaves its output at $0$, so it cannot change the loss.

# Deep Dive: deriving the linear layer's gradients

---

The forward op is, for sample `i`, output unit `j`:

$$\text{out}[i,j] \;=\; \sum_k \text{inp}[i,k]\,w[k,j] \;+\; b[j].$$

We are *given* `out.g[i,j] = ∂L/∂out[i,j]`. Apply the chain rule three times.

### 1. Gradient w.r.t. the bias — `b.g = out.g.sum(0)`

`b[j]` is added to `out[i,j]` for **every sample `i`**, so `∂out[i,j]/∂b[j] = 1`, and

$$\frac{\partial L}{\partial b[j]} \;=\; \sum_i \frac{\partial L}{\partial \text{out}[i,j]}\cdot 1 \;=\; \sum_i \text{out.g}[i,j].$$

Because the bias is *broadcast* across the batch in the forward pass, its gradient is **summed** across the batch in the backward pass — `out.g.sum(0)` collapses the batch dimension `(bs, out) → (out,)`. **Broadcast forward ⇒ sum backward** is a rule you will see again and again.

### 2. Gradient w.r.t. the weights — `w.g = inp.T @ out.g`

`w[k,j]` multiplies `inp[i,k]`, so `∂out[i,j]/∂w[k,j] = inp[i,k]`, giving

$$\frac{\partial L}{\partial w[k,j]} \;=\; \sum_i \text{inp}[i,k]\,\text{out.g}[i,j].$$

That sum over the batch `i` of `inp[i,k]·out.g[i,j]` is exactly the matrix product `inp.T @ out.g`, shape `(in, bs)@(bs, out) = (in, out)` — same shape as `w`. (The notebook's `(inp.unsqueeze(-1) * out.g.unsqueeze(1)).sum(0)` is the same computation written out with broadcasting instead of `@`.)

### 3. Gradient w.r.t. the input — `inp.g = out.g @ w.t()`

`inp[i,k]` feeds **every** output unit `j` (through `w[k,j]`), so we sum its influence over `j`:

$$\frac{\partial L}{\partial \text{inp}[i,k]} \;=\; \sum_j \text{out.g}[i,j]\,w[k,j] \;=\; (\text{out.g}\;@\;w^{\mathsf T})[i,k].$$

Shape `(bs, out)@(out, in) = (bs, in)` — same shape as `inp`. This is the gradient we **hand to the previous layer** as *its* `out.g`. Each gradient comes out the same shape as the thing it's the gradient of — a great built-in sanity check.

# Deep Dive: deriving the *whole* forward + backward pass from scratch

The code below (`forward_and_backward`) is the heart of the lesson. Every single line of it is just the **chain rule** applied step by step. Let's derive all of it from scratch, in plain steps, and at each step point to the exact line of code it becomes.

> **The one rule we keep using.** If a quantity $L$ depends on $u$, and $u$ depends on $v$, then
> $$\frac{\partial L}{\partial v} = \frac{\partial L}{\partial u}\cdot\frac{\partial u}{\partial v}.$$
> "To get the gradient further back, take the gradient you already have and **multiply** by the local derivative of the next step." That's literally all backprop is — done once per layer, in reverse.

---

## 1. Write the network as math (this is the *forward* pass)

Our batch of inputs is a matrix $X$ with one image **per row**: shape `(n, 784)`, where $n$ is the batch size. The four forward lines of code are exactly these four equations:

| Code | Math | Shape |
|---|---|---|
| `l1 = lin(inp, w1, b1)` | $Z_1 = X\,W_1 + b_1$ | `(n, 50)` |
| `l2 = relu(l1)` | $A_1 = \mathrm{ReLU}(Z_1) = \max(0,\,Z_1)$ | `(n, 50)` |
| `out = lin(l2, w2, b2)` | $\hat Y = A_1\,W_2 + b_2$ | `(n, 1)` |
| `loss = diff.pow(2).mean()` | $L = \dfrac{1}{n}\displaystyle\sum_{i=1}^{n}\big(\hat y_i - y_i\big)^2$ | scalar |

where `diff = out[:,0] - targ` is just the error vector $\mathbf{e} = \hat{\mathbf y} - \mathbf y$ of shape `(n,)`.

So the data flows **forward**: $\;X \to Z_1 \to A_1 \to \hat Y \to L.\;$ To train, we need the gradient of $L$ with respect to every weight. We get there by walking that same chain **backward**: $\;L \to \hat Y \to A_1 \to Z_1 \to X.\;$ Each arrow contributes one multiplication.

---

## 2. Step 1 — gradient of the loss w.r.t. the output  →  `out.g = 2.*diff[:,None]/inp.shape[0]`

Start at the very end. The loss for one sample is $\ell_i = (\hat y_i - y_i)^2$, and $L = \frac{1}{n}\sum_i \ell_i$. Differentiate one term with the **power rule** $\frac{d}{dt}t^2 = 2t$ (here $t = \hat y_i - y_i$, and $y_i$ is a constant so its derivative is $0$):

$$\frac{\partial L}{\partial \hat y_i} = \frac{1}{n}\cdot 2\,(\hat y_i - y_i) = \frac{2}{n}\,(\hat y_i - y_i).$$

Stacking over all $n$ samples, $\dfrac{\partial L}{\partial \hat Y} = \dfrac{2}{n}\,(\hat Y - Y) = \dfrac{2}{n}\,\mathbf e.$

That **is** the code: `2. * diff[:,None] / inp.shape[0]`. The `[:, None]` reshapes `(n,)` back to `(n, 1)` so it matches the shape of `out`; `inp.shape[0]` is the $n$ in the $\tfrac{2}{n}$. We store it as `out.g` — the gradient that now starts flowing backward.

---

## 3. Step 2 — push the gradient through linear layer 2  →  `lin_grad(l2, out, w2, b2)`

Now we have $G \equiv \dfrac{\partial L}{\partial \hat Y}$ (`out.g`), and the layer was
$$\hat Y = A_1 W_2 + b_2,\qquad\text{written out per element:}\qquad \hat y_{ij} = \sum_k A_{1,ik}\,W_{2,kj} + b_{2,j}.$$
Here $i$ indexes the **sample** (row), $j$ the **output unit**, $k$ the **hidden unit**. We now derive **each** of the three gradients element by element — nothing skipped. The strategy every time: *(1) find which outputs the quantity touches, (2) write its local derivative, (3) chain-rule sum over all those outputs, (4) recognize the sum as a matrix product.*

### (a) Weights — `w2.g = l2.T @ out.g`

Pick a single weight $W_{2,kj}$. **Which outputs does it touch?** In $\hat y_{ij'} = \sum_{k'} A_{1,ik'}W_{2,k'j'} + b_{2,j'}$, the weight $W_{2,kj}$ shows up **only when $j'=j$** (its own output column), and then its multiplier is $A_{1,ik}$:
$$\frac{\partial \hat y_{ij'}}{\partial W_{2,kj}} = \begin{cases} A_{1,ik} & \text{if } j'=j\\[3pt] 0 & \text{if } j'\neq j.\end{cases}$$
But it affects that one output column for **every sample $i$**. So the chain rule sums over *all* outputs $(i,j')$:
$$\frac{\partial L}{\partial W_{2,kj}} = \sum_{i}\sum_{j'} \underbrace{\frac{\partial L}{\partial \hat y_{ij'}}}_{=\,G_{ij'}}\,\frac{\partial \hat y_{ij'}}{\partial W_{2,kj}} \;=\; \sum_i G_{ij}\,A_{1,ik}\qquad(\text{only } j'=j \text{ survives}).$$
**Step 4 — recognize the matrix product.** By the definition of matrix multiplication, $(A_1^{\top}G)_{kj} = \sum_i (A_1^{\top})_{ki}\,G_{ij} = \sum_i A_{1,ik}\,G_{ij}$ — *exactly* the sum above. Therefore
$$\boxed{\;\frac{\partial L}{\partial W_2} = A_1^{\top} G\;}\quad\Longleftrightarrow\quad \texttt{w2.g = l2.T @ out.g},\qquad \text{shapes } (50,n)\cdot(n,1)=(50,1)=\text{shape of }W_2.$$

### (b) Bias — `b2.g = out.g.sum(0)`

For a single bias $b_{2,j}$: in $\hat y_{ij'}$ it appears only when $j'=j$, with local derivative $1$:
$$\frac{\partial \hat y_{ij'}}{\partial b_{2,j}} = \begin{cases}1 & j'=j\\ 0 & j'\neq j\end{cases}\qquad\Rightarrow\qquad \frac{\partial L}{\partial b_{2,j}} = \sum_i \sum_{j'} G_{ij'}\,\frac{\partial \hat y_{ij'}}{\partial b_{2,j}} = \sum_i G_{ij}.$$
Summing $G$ **down the rows** (over the batch $i$) gives $\dfrac{\partial L}{\partial b_2} = \sum_i G_{i\cdot}$, i.e. `out.g.sum(0)`. *(The bias is **broadcast** across samples in the forward pass ⇒ its gradient is **summed** across samples in the backward pass — the same "broadcast forward ⇒ sum backward" rule from the deep-dive above.)*

### (c) Input to this layer — `l2.g = out.g @ w2.T`

Now a hidden activation $A_{1,ik}$. **Which outputs does it touch?** In its own row $i$ it feeds **every** output unit $j$, with local derivative $\partial \hat y_{ij}/\partial A_{1,ik} = W_{2,kj}$. (It does not touch other rows.) Chain-rule sum over the outputs $j$ it influences:
$$\frac{\partial L}{\partial A_{1,ik}} = \sum_j \frac{\partial L}{\partial \hat y_{ij}}\,\frac{\partial \hat y_{ij}}{\partial A_{1,ik}} = \sum_j G_{ij}\,W_{2,kj}.$$
Recognize the product: $(G\,W_2^{\top})_{ik} = \sum_j G_{ij}\,(W_2^{\top})_{jk} = \sum_j G_{ij}\,W_{2,kj}$ — identical. Therefore
$$\boxed{\;\frac{\partial L}{\partial A_1} = G\,W_2^{\top}\;}\quad\Longleftrightarrow\quad \texttt{l2.g = out.g @ w2.t()},\qquad \text{shapes } (n,1)\cdot(1,50)=(n,50)=\text{shape of }A_1.$$
This `l2.g` is the gradient at the ReLU's **output** $\dfrac{\partial L}{\partial A_1}$, which we hand to Step 3. *(These are the same three rules proved generically in the "deriving the linear layer's gradients" deep-dive above — here written out specifically for layer 2.)*

---

## 4. Step 3 — push through the ReLU  →  `l1.g = (l1 > 0).float() * l2.g`

ReLU acts on each number **independently**: $A_1 = \max(0, Z_1)$. So we only need its one-variable derivative:

$$\text{if } z > 0:\; \mathrm{ReLU}(z)=z \Rightarrow \frac{d}{dz}=1, \qquad\quad \text{if } z < 0:\; \mathrm{ReLU}(z)=0 \Rightarrow \frac{d}{dz}=0.$$

In one symbol, $\mathrm{ReLU}'(z) = \mathbb{1}[z>0]$ (it's $1$ where the unit was "on", $0$ where it was "off"). By the chain rule, elementwise:

$$\frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial A_1}\;\odot\;\mathbb{1}[Z_1>0]$$

($\odot$ means multiply element-by-element.) In code, `(l1 > 0)` builds the mask of $1$s and $0$s and we multiply it by `l2.g` ($=\partial L/\partial A_1$). The gradient **passes through** active units and is **blocked** at units that were off. Result: `l1.g` $= \dfrac{\partial L}{\partial Z_1}$.

---

## 5. Step 4 — push through linear layer 1  →  `lin_grad(inp, l1, w1, b1)`

Same kind of layer, $Z_1 = X W_1 + b_1$, i.e. per element $z_{1,ij} = \sum_k X_{ik}\,W_{1,kj} + b_{1,j}$, now with the incoming gradient $H \equiv \dfrac{\partial L}{\partial Z_1}$ = `l1.g` from Step 3. The derivation is **the exact same three steps** as Step 2 with $X$ in place of $A_1$ and $H$ in place of $G$ — shown in full here so nothing is left implicit.

### (a) Weights — `w1.g = inp.T @ l1.g`
$W_{1,kj}$ touches output $z_{1,ij'}$ only for $j'=j$, with local derivative $X_{ik}$, for every sample $i$:
$$\frac{\partial L}{\partial W_{1,kj}} = \sum_i\sum_{j'} H_{ij'}\,\frac{\partial z_{1,ij'}}{\partial W_{1,kj}} = \sum_i H_{ij}\,X_{ik} = (X^{\top}H)_{kj}\;\;\Rightarrow\;\; \boxed{\dfrac{\partial L}{\partial W_1}=X^{\top}H}\;\Longleftrightarrow\;\texttt{w1.g = inp.T @ l1.g}.$$

### (b) Bias — `b1.g = l1.g.sum(0)`
$b_{1,j}$ touches $z_{1,ij'}$ only for $j'=j$, local derivative $1$, summed over the batch:
$$\frac{\partial L}{\partial b_{1,j}} = \sum_i\sum_{j'} H_{ij'}\,\frac{\partial z_{1,ij'}}{\partial b_{1,j}} = \sum_i H_{ij}\;\;\Rightarrow\;\; \boxed{\dfrac{\partial L}{\partial b_1}=\sum_i H_{i\cdot}}\;\Longleftrightarrow\;\texttt{b1.g = l1.g.sum(0)}.$$

### (c) Input — `inp.g = l1.g @ w1.T`
$X_{ik}$ feeds every output $j$ in its row, local derivative $W_{1,kj}$, summed over $j$:
$$\frac{\partial L}{\partial X_{ik}} = \sum_j H_{ij}\,\frac{\partial z_{1,ij}}{\partial X_{ik}} = \sum_j H_{ij}\,W_{1,kj} = (H\,W_1^{\top})_{ik}\;\;\Rightarrow\;\; \boxed{\dfrac{\partial L}{\partial X}=H\,W_1^{\top}}\;\Longleftrightarrow\;\texttt{inp.g = l1.g @ w1.t()}.$$

The input gradient `inp.g` is rarely used here (we don't train the pixels), but it is exactly what we'd hand to a *previous* layer if there were one — which is precisely how this scheme scales to deep networks.

---

## 6. The whole derivation on one line per step

$$\underbrace{\frac{2}{n}(\hat Y - Y)}_{\text{Step 1: out.g}} \;\xrightarrow[\;\times W_2^\top\;]{\text{Step 2}}\; \frac{\partial L}{\partial A_1} \;\xrightarrow[\;\odot\,\mathbb 1[Z_1>0]\;]{\text{Step 3: ReLU}}\; \frac{\partial L}{\partial Z_1} \;\xrightarrow[\;\times W_1^\top\;]{\text{Step 4}}\; \frac{\partial L}{\partial X}$$

and at **each** linear step we *also* grabbed the weight/bias gradients as `(layer input)ᵀ @ (gradient)` and `(gradient).sum(0)`. Map of math → code:

| Derived result | Code line |
|---|---|
| $\partial L/\partial \hat Y = \tfrac{2}{n}(\hat Y - Y)$ | `out.g = 2.*diff[:,None]/inp.shape[0]` |
| $\partial L/\partial A_1 = (\partial L/\partial \hat Y)\,W_2^\top$ | `l2.g = out.g @ w2.t()` |
| $\partial L/\partial W_2 = A_1^\top(\partial L/\partial \hat Y)$ | `w2.g = l2.T @ out.g` |
| $\partial L/\partial b_2 = \sum_i(\partial L/\partial \hat Y)_{i}$ | `b2.g = out.g.sum(0)` |
| $\partial L/\partial Z_1 = (\partial L/\partial A_1)\odot\mathbb 1[Z_1>0]$ | `l1.g = (l1>0).float()*l2.g` |
| $\partial L/\partial W_1 = X^\top(\partial L/\partial Z_1)$ | `w1.g = inp.T @ l1.g` |
| $\partial L/\partial b_1 = \sum_i(\partial L/\partial Z_1)_{i}$ | `b1.g = l1.g.sum(0)` |
| $\partial L/\partial X = (\partial L/\partial Z_1)\,W_1^\top$ | `inp.g = l1.g @ w1.t()` |

That's it — nothing in the code below is mysterious. It is the chain rule, applied four times, from the loss back to the input. Now read the implementation and watch each comment line up with a row of the table above. 👇

In [ ]:
# ============================================================================
# THE COMPLETE FORWARD AND BACKWARD PASS
# ============================================================================
# This is the heart of neural network training!
# We compute predictions (forward) then compute how to improve (backward).

def forward_and_backward(inp, targ):
    """
    Perform complete forward and backward pass through the network.
    
    Args:
        inp:  Input data (training images)
              Shape: (batch_size, 784) - e.g., (50000, 784)
        
        targ: Target labels (correct digit for each image)
              Shape: (batch_size,) - e.g., (50000,)
    
    After calling this function:
        - w1.g, b1.g: Gradients for first layer parameters
        - w2.g, b2.g: Gradients for second layer parameters
        - inp.g: Gradient with respect to input (rarely used)
    
    The gradients tell us how to adjust each parameter to reduce loss.
    """
    
    # =========================================================================
    # FORWARD PASS: Compute predictions
    # =========================================================================
    # Data flows: input → linear1 → relu → linear2 → loss
    
    # Step 1: First linear layer
    # l1 = inp @ w1 + b1
    # Shape: (50000, 784) @ (784, 50) + (50,) = (50000, 50)
    l1 = lin(inp, w1, b1)
    
    # Step 2: ReLU activation
    # l2 = max(0, l1)
    # Shape: (50000, 50) - same as l1, just with negatives zeroed
    l2 = relu(l1)
    
    # Step 3: Second linear layer (output layer)
    # out = l2 @ w2 + b2
    # Shape: (50000, 50) @ (50, 1) + (1,) = (50000, 1)
    out = lin(l2, w2, b2)
    
    # Step 4: Compute error (difference between prediction and target)
    # diff = prediction - target
    # out[:,0] converts (50000, 1) → (50000,) to match targ shape
    diff = out[:, 0] - targ
    
    # Step 5: Compute MSE loss
    # loss = mean(diff²)
    # This is a single number representing how wrong we are overall
    loss = diff.pow(2).mean()
    
    # =========================================================================
    # BACKWARD PASS: Compute gradients
    # =========================================================================
    # Gradients flow: loss → linear2 → relu → linear1 → input
    # We go in REVERSE order!
    
    # Step 1: Gradient of MSE loss with respect to output
    # ─────────────────────────────────────────────────────────────────────────
    # d(loss)/d(out) = d/d(out)[mean((out - targ)²)]
    #                = 2 × (out - targ) / n
    #
    # The [:, None] converts (50000,) → (50000, 1) to match out's shape
    # inp.shape[0] is the batch size (n) for averaging
    out.g = 2. * diff[:, None] / inp.shape[0]
    
    # Step 2: Gradient through second linear layer
    # ─────────────────────────────────────────────────────────────────────────
    # This sets: l2.g, w2.g, b2.g
    # l2.g = out.g @ w2.T  (gradient flows back to relu output)
    # w2.g = l2.T @ out.g  (gradient for layer 2 weights)
    # b2.g = sum(out.g)    (gradient for layer 2 bias)
    lin_grad(l2, out, w2, b2)
    
    # Step 3: Gradient through ReLU
    # ─────────────────────────────────────────────────────────────────────────
    # ReLU Equation: f(x) = max(0, x)
    # Derivation:
    #   Case 1: x > 0 -> f(x) = x -> d/dx(f(x)) = 1
    #   Case 2: x < 0 -> f(x) = 0 -> d/dx(f(x)) = 0
    #
    # By Chain Rule: d(loss)/d(l1) = d(loss)/d(l2) * d(l2)/d(l1)
    #                              = l2.g * (1.0 if l1 > 0 else 0.0)
    #
    # (l1 > 0) creates a boolean mask; .float() converts to 1.0/0.0
    l1.g = (l1 > 0).float() * l2.g
    
    # Step 4: Gradient through first linear layer
    # ─────────────────────────────────────────────────────────────────────────
    # This sets: inp.g, w1.g, b1.g
    # inp.g = l1.g @ w1.T  (gradient flows back to input - rarely used)
    # w1.g = inp.T @ l1.g  (gradient for layer 1 weights)
    # b1.g = sum(l1.g)     (gradient for layer 1 bias)
    lin_grad(inp, l1, w1, b1)

print("forward_and_backward function defined!")
print("This computes both the loss AND all gradients in one pass.")

## How is `l2.g` available after `lin_grad(l2, out, w2, b2)`?  (pass-by-reference + the `.g` trick)

Inside `forward_and_backward` we call `lin_grad(l2, out, w2, b2)`, and `lin_grad` only ever sets `inp.g` internally. Yet the very next line happily reads `l2.g`:

```python
lin_grad(l2, out, w2, b2)        # sets inp.g, w.g, b.g  -- but inp, not l2 ...?
l1.g = (l1 > 0).float() * l2.g   # ... so how does l2.g exist here?
```

Two Python facts explain it.

### 1. `inp` is just a nickname for `l2` (pass-by-object-reference)

Arguments bind to parameters **by position**, and Python passes the *same object*, never a copy. So for this particular call:

| nickname inside `lin_grad` | the object it actually refers to |
|---|---|
| `inp` | `l2` |
| `out` | `out` |
| `w`   | `w2` |
| `b`   | `b2` |

Substitute those into `lin_grad`'s body and, for this call, it literally runs:

```python
l2.g = out.g @ w2.t()       # <- THIS is the line that creates l2.g
w2.g = (l2.unsqueeze(-1) * out.g.unsqueeze(1)).sum(0)
b2.g = out.g.sum(0)
```

`inp` was only a temporary name for `l2`, so `inp.g = …` is the same as `l2.g = …`.

### 2. Setting an attribute mutates the object — and scope controls *names*, not *objects*

You might worry: "`l2.g` was set *inside* `lin_grad`, so isn't it discarded when the function returns?" No — and this is the crux:

- When a function returns, Python throws away its **local names** (`inp`, `out`, `w`, `b`, and any `x = …` you make inside). It does **not** throw away **objects** — an object lives as long as something still references it. `l2` was created here in `forward_and_backward` and is still held here, so its object survives whatever `lin_grad` does.
- `inp.g = …` does **not** create a local variable. It reaches *into the object* and bolts on an attribute `g`. That attribute lives **on the object**, not in `lin_grad`'s namespace. So when `lin_grad`'s locals are cleared, the object (held outside as `l2`) keeps its new `.g`. **Where you write the attribute is not where it lives.**

These eight lines show **names** (which don't leak) versus **attributes** (which do):

```python
class Thing: pass
a = Thing()                 # object created out here

def f(x):
    x.g = 99                # ATTRIBUTE on the object  -> stored ON the object
    local = 7               # LOCAL VARIABLE (a name)  -> dies when f returns

f(a)
print(a.g)                  # 99    survived: it lives on the object 'a'
print(local)                # NameError: 'local' is not defined  (the name was local)
```

Map it back: `f` is `lin_grad`, `x` is `inp`, `a` is `l2`. The line `x.g = 99` is `inp.g = …`, which (since `x` *is* `a`) sets `a.g` — i.e. `l2.g`. The `local = 7` line is the case your intuition was right about: *that* really is discarded on return. The **only** reason `.g` survives is that it was written **into an object owned by the outer scope**, not stored as a local name.

### The backward pass as a relay

Each `lin_grad` call writes the `.g` that the next step reads:

```python
out.g = 2.*diff[:,None]/n      # write .g onto the output object
lin_grad(l2, out, w2, b2)      # reads out.g; writes l2.g, w2.g, b2.g   <- l2.g now exists
l1.g = (l1>0).float() * l2.g   # reads the freshly-written l2.g (the ReLU gate)
lin_grad(inp, l1, w1, b1)      # reads l1.g; writes inp.g, w1.g, b1.g
```

The gradients are handed from one step to the next by **stashing them as attributes on the very tensors that flowed through the forward pass** — a tidy trick that avoids returning and threading values around. (PyTorch's real autograd fills the `.grad` attribute on leaf tensors the same in-place way during `loss.backward()`.)

> **In one line:** local *names* are scoped and vanish on return, but an *attribute set on an object* lives on that object — and `l2`'s object belongs to `forward_and_backward`, so the `.g` written into it from inside `lin_grad` is still there afterwards.

**What does the code above do?**

`forward_and_backward` is the whole training signal in one function. **Forward:** `l1 = lin(inp,w1,b1) → l2 = relu(l1) → out = lin(l2,w2,b2) → loss = mse`. **Backward (reverse order):** seed `out.g = 2·(out−y)/n`, then `lin_grad` through layer 2, then the ReLU mask `l1.g = (l1>0).float()*l2.g`, then `lin_grad` through layer 1. Afterwards every parameter carries its `.g`. Note the backward steps are the forward steps **read bottom-to-top**.

### 🎮 Interactive: one full training step

Press **Step →** (or **Auto**) to send data forward through `784 → 50 → 1` and compute the loss, then watch the gradients flow back through each layer with the exact lines of code. Forward computes *values*; backward computes *gradients* — mirror images of each other.

In [ ]:
# ============================================================================
# INTERACTIVE (embedded figure) -- run this cell. Self-contained iframe;
# works in Jupyter, Colab, and the exported HTML. Re-run to reset it.
# ============================================================================
from IPython.display import HTML
HTML('<iframe src="data:text/html;base64,PCFET0NUWVBFIGh0bWw+PGh0bWwgbGFuZz0iZW4iPjxoZWFkPjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+Rm9yd2FyZCBhbmQgYmFja3dhcmQ8L3RpdGxlPgo8c3R5bGU+CiAgOnJvb3R7LS1pbms6IzI1MzI0YTstLXNvZnQ6IzVhNmI4NjstLWxpbmU6I2RkZTVmMjstLWluOiNmNjkyMWU7LS1vdXQ6IzE2YTNhMzstLWFjOiM3YjVjZDY7LS1iZzojZWVmM2ZiO30KICAqe2JveC1zaXppbmc6Ym9yZGVyLWJveDt9aHRtbCxib2R5e21hcmdpbjowO2ZvbnQtZmFtaWx5OiJTZWdvZSBVSSIsc3lzdGVtLXVpLHNhbnMtc2VyaWY7Y29sb3I6dmFyKC0taW5rKTt9CiAgYm9keXtiYWNrZ3JvdW5kOnJhZGlhbC1ncmFkaWVudCg5MDBweCA0MDBweCBhdCAzMCUgLTEwJSwjZjNmN2ZmLHZhcigtLWJnKSksdmFyKC0tYmcpO3BhZGRpbmc6MTRweDt9CiAgaDF7Zm9udC1zaXplOjE3cHg7bWFyZ2luOjAgMCAycHg7dGV4dC1hbGlnbjpjZW50ZXI7fQogIC5zdWJ7dGV4dC1hbGlnbjpjZW50ZXI7Y29sb3I6dmFyKC0tc29mdCk7Zm9udC1zaXplOjEycHg7bWFyZ2luOjAgMCAxMnB4O30KICAucGlwZXtkaXNwbGF5OmZsZXg7YWxpZ24taXRlbXM6c3RyZXRjaDtqdXN0aWZ5LWNvbnRlbnQ6Y2VudGVyO2dhcDowO2ZsZXgtd3JhcDp3cmFwO30KICAuc3RhZ2V7YmFja2dyb3VuZDojZmZmO2JvcmRlcjoycHggc29saWQgdmFyKC0tbGluZSk7Ym9yZGVyLXJhZGl1czoxMnB4O3BhZGRpbmc6OHB4IDlweDttaW4td2lkdGg6OTZweDt0ZXh0LWFsaWduOmNlbnRlcjtib3gtc2hhZG93OjAgNnB4IDE2cHggcmdiYSgxMjMsOTIsMjE0LC4wOCk7fQogIC5zdGFnZSAubntmb250LXdlaWdodDo3MDA7Zm9udC1zaXplOjEycHg7fQogIC5zdGFnZSAuc3tmb250LWZhbWlseTptb25vc3BhY2U7Zm9udC1zaXplOjEwLjVweDtjb2xvcjp2YXIoLS1zb2Z0KTttYXJnaW4tdG9wOjJweDt9CiAgLnN0YWdlIC5vcHtmb250LWZhbWlseTptb25vc3BhY2U7Zm9udC1zaXplOjExcHg7Y29sb3I6dmFyKC0tYWMpO21hcmdpbi10b3A6M3B4O30KICAuY29ubnt3aWR0aDozMHB4O2FsaWduLXNlbGY6Y2VudGVyO3RleHQtYWxpZ246Y2VudGVyO2NvbG9yOnZhcigtLWxpbmUpO2ZvbnQtc2l6ZToxOHB4O30KICAuc3RhZ2UuZndke2JvcmRlci1jb2xvcjp2YXIoLS1pbik7Ym94LXNoYWRvdzowIDAgMCAzcHggcmdiYSgyNDYsMTQ2LDMwLC4xOCk7fQogIC5zdGFnZS5id2R7Ym9yZGVyLWNvbG9yOnZhcigtLW91dCk7Ym94LXNoYWRvdzowIDAgMCAzcHggcmdiYSgyMiwxNjMsMTYzLC4xOCk7fQogIC5wYW5lbHtiYWNrZ3JvdW5kOiNmZmY7Ym9yZGVyOjFweCBzb2xpZCB2YXIoLS1saW5lKTtib3JkZXItcmFkaXVzOjEycHg7cGFkZGluZzoxMHB4IDEzcHg7Zm9udC1zaXplOjEzcHg7bGluZS1oZWlnaHQ6MS41NTttYXJnaW4tdG9wOjEycHg7bWluLWhlaWdodDo3MHB4O30KICAucGFuZWwgYntjb2xvcjp2YXIoLS1hYyk7fQogIC5lcXtmb250LWZhbWlseTptb25vc3BhY2U7YmFja2dyb3VuZDojZjRmN2ZkO2JvcmRlci1yYWRpdXM6OHB4O3BhZGRpbmc6NnB4IDlweDttYXJnaW4tdG9wOjZweDtmb250LXNpemU6MTJweDtvdmVyZmxvdy14OmF1dG87fQogIC5jdHJ7ZGlzcGxheTpmbGV4O2dhcDo4cHg7anVzdGlmeS1jb250ZW50OmNlbnRlcjttYXJnaW4tdG9wOjEycHg7ZmxleC13cmFwOndyYXA7fQogIGJ1dHRvbntmb250LWZhbWlseTppbmhlcml0O2ZvbnQtd2VpZ2h0OjcwMDtmb250LXNpemU6MTNweDtib3JkZXI6bm9uZTtib3JkZXItcmFkaXVzOjlweDtwYWRkaW5nOjhweCAxNXB4O2N1cnNvcjpwb2ludGVyO30KICAubmV4dHtiYWNrZ3JvdW5kOnZhcigtLWFjKTtjb2xvcjojZmZmO30uYXV0b3tiYWNrZ3JvdW5kOnZhcigtLWluKTtjb2xvcjojZmZmO30ucmVzZXR7YmFja2dyb3VuZDojZWNlN2ZiO2NvbG9yOnZhcigtLWFjKTt9CiAgLmJhcntoZWlnaHQ6NnB4O2JvcmRlci1yYWRpdXM6NHB4O2JhY2tncm91bmQ6I2VlZjJmODttYXJnaW46MTBweCBhdXRvIDA7bWF4LXdpZHRoOjY0MHB4O292ZXJmbG93OmhpZGRlbjt9CiAgLmJhcj5kaXZ7aGVpZ2h0OjEwMCU7d2lkdGg6MDtiYWNrZ3JvdW5kOmxpbmVhci1ncmFkaWVudCg5MGRlZyx2YXIoLS1pbiksdmFyKC0tb3V0KSk7dHJhbnNpdGlvbjp3aWR0aCAuMjVzO30KICAubGVnZW5ke3RleHQtYWxpZ246Y2VudGVyO2ZvbnQtc2l6ZToxMXB4O2NvbG9yOnZhcigtLXNvZnQpO21hcmdpbi10b3A6OHB4O30KICAubGVnZW5kIC5he2NvbG9yOnZhcigtLWluKTtmb250LXdlaWdodDo3MDA7fS5sZWdlbmQgLmJ7Y29sb3I6dmFyKC0tb3V0KTtmb250LXdlaWdodDo3MDA7fQo8L3N0eWxlPjwvaGVhZD48Ym9keT4KPGgxPk9uZSB0cmFpbmluZyBzdGVwOiBmb3J3YXJkIHRoZW4gYmFja3dhcmQ8L2gxPgo8cCBjbGFzcz0ic3ViIj5XYXRjaCBkYXRhIGZsb3cgcmlnaHQgKGNvbXB1dGUgdGhlIHByZWRpY3Rpb24gJmFtcDsgbG9zcyksIHRoZW4gZ3JhZGllbnRzIGZsb3cgbGVmdCAoaG93IHRvIGZpeCBldmVyeSB3ZWlnaHQpLiBTYW1lIG5ldHdvcmsgYXMgdGhlIG5vdGVib29rOiA3ODQg4oaSIDUwIOKGkiAxLjwvcD4KPGRpdiBjbGFzcz0icGlwZSI+CiAgPGRpdiBjbGFzcz0ic3RhZ2UiIGlkPSJzMCI+PGRpdiBjbGFzcz0ibiI+aW5wdXQgeDwvZGl2PjxkaXYgY2xhc3M9InMiPihicywgNzg0KTwvZGl2PjxkaXYgY2xhc3M9Im9wIj4mbmJzcDs8L2Rpdj48L2Rpdj4KICA8ZGl2IGNsYXNzPSJjb25uIj7ihpI8L2Rpdj4KICA8ZGl2IGNsYXNzPSJzdGFnZSIgaWQ9InMxIj48ZGl2IGNsYXNzPSJuIj5MaW4gMTwvZGl2PjxkaXYgY2xhc3M9InMiPihicywgNTApPC9kaXY+PGRpdiBjbGFzcz0ib3AiPnhAdzErYjE8L2Rpdj48L2Rpdj4KICA8ZGl2IGNsYXNzPSJjb25uIj7ihpI8L2Rpdj4KICA8ZGl2IGNsYXNzPSJzdGFnZSIgaWQ9InMyIj48ZGl2IGNsYXNzPSJuIj5SZUxVPC9kaXY+PGRpdiBjbGFzcz0icyI+KGJzLCA1MCk8L2Rpdj48ZGl2IGNsYXNzPSJvcCI+bWF4KDAswrcpPC9kaXY+PC9kaXY+CiAgPGRpdiBjbGFzcz0iY29ubiI+4oaSPC9kaXY+CiAgPGRpdiBjbGFzcz0ic3RhZ2UiIGlkPSJzMyI+PGRpdiBjbGFzcz0ibiI+TGluIDI8L2Rpdj48ZGl2IGNsYXNzPSJzIj4oYnMsIDEpPC9kaXY+PGRpdiBjbGFzcz0ib3AiPsK3QHcyK2IyPC9kaXY+PC9kaXY+CiAgPGRpdiBjbGFzcz0iY29ubiI+4oaSPC9kaXY+CiAgPGRpdiBjbGFzcz0ic3RhZ2UiIGlkPSJzNCI+PGRpdiBjbGFzcz0ibiI+TVNFIGxvc3M8L2Rpdj48ZGl2IGNsYXNzPSJzIj5zY2FsYXI8L2Rpdj48ZGl2IGNsYXNzPSJvcCI+KMK34oiSeSnCsjwvZGl2PjwvZGl2Pgo8L2Rpdj4KPGRpdiBjbGFzcz0iYmFyIj48ZGl2IGlkPSJwcm9nIj48L2Rpdj48L2Rpdj4KPGRpdiBjbGFzcz0icGFuZWwiIGlkPSJwYW5lbCI+UHJlc3MgPGI+U3RlcCDihpI8L2I+IChvciA8Yj5BdXRvPC9iPikgdG8gcnVuIG9uZSBmb3J3YXJkIHBhc3MsIHRoZW4gb25lIGJhY2t3YXJkIHBhc3MuPC9kaXY+CjxkaXYgY2xhc3M9ImN0ciI+CiAgPGJ1dHRvbiBjbGFzcz0ibmV4dCIgaWQ9InN0ZXAiPlN0ZXAg4oaSPC9idXR0b24+CiAgPGJ1dHRvbiBjbGFzcz0iYXV0byIgaWQ9ImF1dG8iPuKPqSBBdXRvPC9idXR0b24+CiAgPGJ1dHRvbiBjbGFzcz0icmVzZXQiIGlkPSJyZXNldCI+4oa6IFJlc2V0PC9idXR0b24+CjwvZGl2Pgo8ZGl2IGNsYXNzPSJsZWdlbmQiPjxzcGFuIGNsYXNzPSJhIj5mb3J3YXJkOiBjb21wdXRlIHZhbHVlcyDihpI8L3NwYW4+ICZuYnNwO8K3Jm5ic3A7IDxzcGFuIGNsYXNzPSJiIj7ihpAgYmFja3dhcmQ6IGNvbXB1dGUgZ3JhZGllbnRzPC9zcGFuPjwvZGl2Pgo8c2NyaXB0Pgpjb25zdCAkPWlkPT5kb2N1bWVudC5nZXRFbGVtZW50QnlJZChpZCk7Ci8vIHN0ZXBzOiAwIHJlYWR5OyAxLi41IGZvcndhcmQgKGxpZ2h0IHMwLi5zNCk7IDYuLjEwIGJhY2t3YXJkIChzNC4uczApCmNvbnN0IEZXPVsKIHtpOjAsdDonPGI+Rm9yd2FyZCAxLzUuPC9iPiBTdGFydCB3aXRoIHRoZSBpbnB1dCBiYXRjaCA8Yj54PC9iPiwgc2hhcGUgKGJzLCA3ODQpLid9LAoge2k6MSx0Oic8Yj5Gb3J3YXJkIDIvNSDigJQgTGluZWFyIDEuPC9iPiA8Y29kZT5sMSA9IHggQCB3MSArIGIxPC9jb2RlPjogbWl4IDc4NCBwaXhlbHMgaW50byA1MCBoaWRkZW4gZmVhdHVyZXMuPGRpdiBjbGFzcz0iZXEiPihicyw3ODQpIEAgKDc4NCw1MCkgKyAoNTAsKSDihpIgKGJzLDUwKTwvZGl2Pid9LAoge2k6Mix0Oic8Yj5Gb3J3YXJkIDMvNSDigJQgUmVMVS48L2I+IDxjb2RlPmwyID0gbWF4KDAsIGwxKTwvY29kZT46IGFkZCBub24tbGluZWFyaXR5LCB6ZXJvaW5nIG5lZ2F0aXZlcy4nfSwKIHtpOjMsdDonPGI+Rm9yd2FyZCA0LzUg4oCUIExpbmVhciAyLjwvYj4gPGNvZGU+b3V0ID0gbDIgQCB3MiArIGIyPC9jb2RlPjogY29sbGFwc2UgNTAgZmVhdHVyZXMgdG8gMSBudW1iZXIuPGRpdiBjbGFzcz0iZXEiPihicyw1MCkgQCAoNTAsMSkgKyAoMSwpIOKGkiAoYnMsMSk8L2Rpdj4nfSwKIHtpOjQsdDonPGI+Rm9yd2FyZCA1LzUg4oCUIExvc3MuPC9iPiA8Y29kZT5sb3NzID0gKChvdXTiiJJ5KcKyKS5tZWFuKCk8L2NvZGU+OiBvbmUgc2NhbGFyIG1lYXN1cmluZyBob3cgd3Jvbmcgd2UgYXJlLid9Cl07CmNvbnN0IEJXPVsKIHtpOjQsdDonPGI+QmFja3dhcmQgMS81LjwvYj4gU2VlZCB0aGUgZ3JhZGllbnQgYXQgdGhlIGxvc3M6IDxjb2RlPm91dC5nID0gMsK3KG91dOKIknkpL248L2NvZGU+LiAo4oiCbG9zcy/iiIJvdXQuKSd9LAoge2k6Myx0Oic8Yj5CYWNrd2FyZCAyLzUg4oCUIHRocm91Z2ggTGluZWFyIDIuPC9iPiA8Y29kZT53Mi5nID0gbDIuVEBvdXQuZzwvY29kZT4sIDxjb2RlPmIyLmcgPSBvdXQuZy5zdW0oMCk8L2NvZGU+LCBhbmQgcGFzcyA8Y29kZT5sMi5nID0gb3V0LmdAdzIuVDwvY29kZT4gZnVydGhlciBiYWNrLid9LAoge2k6Mix0Oic8Yj5CYWNrd2FyZCAzLzUg4oCUIHRocm91Z2ggUmVMVS48L2I+IDxjb2RlPmwxLmcgPSAobDEmZ3Q7MCkuZmxvYXQoKSAqIGwyLmc8L2NvZGU+OiBncmFkaWVudCBvbmx5IGZsb3dzIHdoZXJlIHRoZSB1bml0IHdhcyBvbi4nfSwKIHtpOjEsdDonPGI+QmFja3dhcmQgNC81IOKAlCB0aHJvdWdoIExpbmVhciAxLjwvYj4gPGNvZGU+dzEuZyA9IHguVEBsMS5nPC9jb2RlPiwgPGNvZGU+YjEuZyA9IGwxLmcuc3VtKDApPC9jb2RlPi4nfSwKIHtpOjAsdDonPGI+QmFja3dhcmQgNS81IOKAlCBkb25lLjwvYj4gRXZlcnkgcGFyYW1ldGVyIG5vdyBoYXMgYSA8Y29kZT4uZzwvY29kZT4uIEFuIG9wdGltaXplciB3b3VsZCBkbyA8Y29kZT53IOKIkj0gbHLCt3cuZzwvY29kZT4uIFRoYXQgaXMgb25lIHRyYWluaW5nIHN0ZXAhJ30KXTsKbGV0IHBoYXNlPTAsdGltZXI9bnVsbDsKZnVuY3Rpb24gY2xlYXJBbGwoKXtmb3IobGV0IGk9MDtpPDU7aSsrKXskKCdzJytpKS5jbGFzc0xpc3QucmVtb3ZlKCdmd2QnLCdid2QnKTt9fQpmdW5jdGlvbiByZW5kZXIoKXsKICBjbGVhckFsbCgpOwogIGxldCBtc2c9J1ByZXNzIDxiPlN0ZXAg4oaSPC9iPiB0byBiZWdpbi4nOwogIGlmKHBoYXNlPj0xJiZwaGFzZTw9NSl7Y29uc3QgZj1GV1twaGFzZS0xXTskKCdzJytmLmkpLmNsYXNzTGlzdC5hZGQoJ2Z3ZCcpO21zZz1mLnQ7fQogIGlmKHBoYXNlPj02JiZwaGFzZTw9MTApe2NvbnN0IGI9QldbcGhhc2UtNl07JCgncycrYi5pKS5jbGFzc0xpc3QuYWRkKCdid2QnKTttc2c9Yi50O30KICAkKCdwYW5lbCcpLmlubmVySFRNTD1tc2c7CiAgJCgncHJvZycpLnN0eWxlLndpZHRoPShwaGFzZS8xMCoxMDApKyclJzsKICAkKCdzdGVwJykudGV4dENvbnRlbnQ9cGhhc2U+PTEwPydEb25lJzonU3RlcCDihpInOyQoJ3N0ZXAnKS5kaXNhYmxlZD1waGFzZT49MTA7Cn0KZnVuY3Rpb24gc3RlcCgpe2lmKHBoYXNlPDEwKXtwaGFzZSsrO3JlbmRlcigpO31lbHNlIHN0b3BBdXRvKCk7fQpmdW5jdGlvbiBzdG9wQXV0bygpe2NsZWFySW50ZXJ2YWwodGltZXIpO3RpbWVyPW51bGw7JCgnYXV0bycpLnRleHRDb250ZW50PSfij6kgQXV0byc7fQokKCdzdGVwJykub25jbGljaz0oKT0+e3N0b3BBdXRvKCk7c3RlcCgpO307CiQoJ2F1dG8nKS5vbmNsaWNrPSgpPT57aWYodGltZXIpe3N0b3BBdXRvKCk7cmV0dXJuO31pZihwaGFzZT49MTApcGhhc2U9MDskKCdhdXRvJykudGV4dENvbnRlbnQ9J+KPuCBQYXVzZSc7dGltZXI9c2V0SW50ZXJ2YWwoKCk9PntzdGVwKCk7aWYocGhhc2U+PTEwKXN0b3BBdXRvKCk7fSw4NTApO307CiQoJ3Jlc2V0Jykub25jbGljaz0oKT0+e3N0b3BBdXRvKCk7cGhhc2U9MDtyZW5kZXIoKTt9OwpyZW5kZXIoKTsKPC9zY3JpcHQ+PC9ib2R5PjwvaHRtbD4K" style="width:100%; height:420px; border:1px solid #dde5f2; border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);" loading="lazy" title="Forward and backward passes"></iframe>')


In [ ]:
# ============================================================================
# RUN THE FORWARD AND BACKWARD PASS
# ============================================================================
# Let's execute our backpropagation on the training data!
# After this call, all parameters (w1, b1, w2, b2) will have gradients stored.

forward_and_backward(x_train, y_train)

print("Forward and backward pass complete!")
print()
print("Now each parameter has a .g attribute containing its gradient:")
print(f"  w1.g shape: {w1.g.shape} - gradient for first layer weights")
print(f"  b1.g shape: {b1.g.shape} - gradient for first layer biases")
print(f"  w2.g shape: {w2.g.shape} - gradient for second layer weights")
print(f"  b2.g shape: {b2.g.shape} - gradient for second layer biases")
print()
print("These gradients tell us how to adjust each parameter to reduce loss!")

In [ ]:
# ============================================================================
# SAVE GRADIENTS FOR LATER VERIFICATION
# ============================================================================
# We'll save copies of our manually computed gradients so we can compare them
# with PyTorch's automatic differentiation (autograd) to verify correctness.

def get_grad(x):
    """
    Get a copy of the gradient stored in x.g
    
    Args:
        x: A tensor with a .g attribute containing its gradient
    
    Returns:
        A clone (copy) of the gradient tensor
        
    Why clone? Without cloning, we'd just get a reference to the same tensor.
    If we later modify x.g, our "saved" gradient would change too!
    """
    return x.g.clone()

# List of tensors we want to check gradients for
chks = w1, w2, b1, b2, x_train

# Save copies of all gradients
# map() applies get_grad to each tensor in chks
# tuple() converts the map result to a tuple
grads = w1g, w2g, b1g, b2g, ig = tuple(map(get_grad, chks))

print("Gradients saved for verification!")
print()
print("Saved gradients:")
print(f"  w1g: gradient for w1, shape {w1g.shape}")
print(f"  w2g: gradient for w2, shape {w2g.shape}")
print(f"  b1g: gradient for b1, shape {b1g.shape}")
print(f"  b2g: gradient for b2, shape {b2g.shape}")
print(f"  ig:  gradient for input, shape {ig.shape}")

### Verifying Our Gradients with PyTorch Autograd

We implemented backpropagation manually - but did we do it correctly? Let's verify by comparing with **PyTorch's automatic differentiation** (autograd).

PyTorch can automatically compute gradients for any computation! It does this by:
1. **Recording operations**: When you compute with tensors that have `requires_grad=True`, PyTorch builds a computation graph
2. **Backward pass**: Calling `.backward()` on a scalar computes all gradients automatically

This is like having a calculator that can differentiate any function!

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                     MANUAL vs AUTOGRAD                                      │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   MANUAL (what we did):                                                     │
│   ─────────────────────                                                     │
│   • We derived gradient formulas by hand                                    │
│   • We implemented them explicitly in code                                  │
│   • Educational but tedious for complex networks                            │
│                                                                             │
│   AUTOGRAD (PyTorch magic):                                                 │
│   ─────────────────────────                                                 │
│   • PyTorch tracks all operations                                           │
│   • Automatically applies chain rule                                        │
│   • Works for ANY computation!                                              │
│                                                                             │
│   Both should give the SAME gradients!                                      │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================================
# PREPARE TENSORS FOR PYTORCH AUTOGRAD
# ============================================================================
# To use PyTorch's autograd, we need tensors with requires_grad=True.
# This tells PyTorch to track all operations on these tensors.

def mkgrad(x):
    """
    Create a copy of tensor x that tracks gradients.
    
    Args:
        x: Any tensor
    
    Returns:
        A clone of x with requires_grad=True
    
    What requires_grad=True does:
        - PyTorch will record all operations involving this tensor
        - We can later call .backward() to compute gradients
        - Gradients are stored in the .grad attribute (not .g like we did)
    """
    return x.clone().requires_grad_(True)

# Create gradient-tracking copies of all our tensors
# These are NEW tensors, independent of our original w1, w2, etc.
ptgrads = w12, w22, b12, b22, xt2 = tuple(map(mkgrad, chks))

print("Created gradient-tracking tensor copies:")
print(f"  w12 (copy of w1): requires_grad = {w12.requires_grad}")
print(f"  w22 (copy of w2): requires_grad = {w22.requires_grad}")
print(f"  b12 (copy of b1): requires_grad = {b12.requires_grad}")
print(f"  b22 (copy of b2): requires_grad = {b22.requires_grad}")
print(f"  xt2 (copy of x_train): requires_grad = {xt2.requires_grad}")

In [ ]:
# ============================================================================
# FORWARD PASS WITH AUTOGRAD-ENABLED TENSORS
# ============================================================================
# We define a forward function using our autograd-enabled tensors.
# This is identical to our earlier forward pass, but uses the new tensors.

def forward(inp, targ):
    """
    Forward pass using autograd-enabled tensors.
    
    Args:
        inp:  Input tensor with requires_grad=True
        targ: Target labels
    
    Returns:
        The MSE loss (a scalar tensor)
    
    Note: We use the autograd tensor copies (w12, b12, w22, b22)
          instead of the originals (w1, b1, w2, b2).
    """
    # First linear layer using autograd copies
    l1 = lin(inp, w12, b12)
    
    # ReLU activation
    l2 = relu(l1)
    
    # Second linear layer
    out = lin(l2, w22, b22)
    
    # Return MSE loss
    return mse(out, targ)

print("Forward function for autograd defined!")
print("Uses same computation, but with gradient-tracking tensors.")

In [ ]:
# ============================================================================
# RUN AUTOGRAD BACKWARD PASS
# ============================================================================
# Now we run the forward pass and let PyTorch compute gradients automatically!

# Step 1: Run forward pass (PyTorch builds computation graph behind the scenes)
loss = forward(xt2, y_train)

# Step 2: Compute all gradients automatically with .backward()
# This single call computes gradients for ALL tensors with requires_grad=True!
# PyTorch traverses the computation graph in reverse, applying the chain rule.
loss.backward()

print(f"Loss: {loss.item():.2f}")
print()
print("PyTorch has now computed gradients automatically!")
print("Gradients are stored in the .grad attribute of each tensor:")
print(f"  w12.grad shape: {w12.grad.shape}")
print(f"  w22.grad shape: {w22.grad.shape}")
print(f"  b12.grad shape: {b12.grad.shape}")
print(f"  b22.grad shape: {b22.grad.shape}")

**What does the code above do?**

Runs the **same** forward computation with `requires_grad=True` tensors and calls `loss.backward()`. PyTorch recorded every operation into a computation graph and now walks it backward automatically — doing exactly what our hand-written `forward_and_backward` did, but derived for us. The next cells `test_close` our `.g` values against PyTorch's `.grad`; they match, which proves our by-hand calculus was right.

In [ ]:
# ============================================================================
# COMPARE MANUAL GRADIENTS VS AUTOGRAD GRADIENTS
# ============================================================================
# The moment of truth! Do our manually computed gradients match PyTorch's?
#
# test_close(a, b, eps=0.01) checks if tensors a and b are approximately equal
# - "Approximately" because floating-point arithmetic can have tiny differences
# - eps=0.01 means we allow differences up to 1%

# Compare each gradient:
# - grads contains our manually computed gradients (w1g, w2g, b1g, b2g, ig)
# - ptgrads contains autograd tensors, with .grad storing PyTorch's gradients

for a, b in zip(grads, ptgrads):
    test_close(a, b.grad, eps=0.01)

# If no errors were raised, all tests passed!
print("✓ All gradient tests passed!")
print()
print("Our manually computed gradients match PyTorch's autograd!")
print()
print("This proves our backpropagation implementation is CORRECT!")
print()
print("Comparison:")
print("  Manual w1.g  ≈  PyTorch w12.grad  ✓")
print("  Manual w2.g  ≈  PyTorch w22.grad  ✓")
print("  Manual b1.g  ≈  PyTorch b12.grad  ✓")
print("  Manual b2.g  ≈  PyTorch b22.grad  ✓")
print("  Manual inp.g ≈  PyTorch xt2.grad  ✓")

---
## Part 3: Refactoring - Object-Oriented Design

Our `forward_and_backward` function works, but it's hard to extend. What if we want:
- A deeper network with more layers?
- Different activation functions?
- Different layer types (convolution, normalization, etc.)?

The solution: **Encapsulate each layer in a class** that knows how to:
1. **Forward**: Compute its output given input
2. **Backward**: Compute gradients given the gradient from the next layer

This is how PyTorch and all deep learning frameworks work!

### Benefits of Object-Oriented Layers

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    FROM FUNCTIONS TO CLASSES                                │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   BEFORE (Functions):                                                       │
│   ───────────────────                                                       │
│   l1 = lin(inp, w1, b1)       # Must track all intermediate values         │
│   l2 = relu(l1)               # Must pass correct variables to each        │
│   out = lin(l2, w2, b2)       # Backward code is separate and fragile      │
│                                                                             │
│   AFTER (Classes):                                                          │
│   ─────────────────                                                         │
│   layer1 = Lin(w1, b1)        # Each layer is self-contained               │
│   relu = Relu()               # Layer stores its own state                 │
│   layer2 = Lin(w2, b2)        # Forward and backward are bundled together  │
│                                                                             │
│   # Forward: just chain the layers                                          │
│   x = layer1(x)                                                             │
│   x = relu(x)                                                               │
│   x = layer2(x)                                                             │
│                                                                             │
│   # Backward: reverse order                                                 │
│   layer2.backward()                                                         │
│   relu.backward()                                                           │
│   layer1.backward()                                                         │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Layers as Classes

#### The ReLU Layer Class

Let's start with the simplest layer: ReLU. It has no learnable parameters, just applies max(0, x).

# Deep Dive: the layer-as-object pattern (`__call__` + saved state)

---

## The problem it solves

`forward_and_backward` works but it's a monolith: the forward and backward logic for each layer are split apart and wired together by hand, and intermediate values (`l1`, `l2`, `out`) live as loose local variables. Adding or reordering a layer means editing the function in two places. We want each layer to be a **self-contained object that knows both directions**.

## The pattern

Each layer becomes a class with two methods:

- **`__call__(self, inp)`** — runs the forward op. Python calls this when you write `layer(x)` (so an instance *behaves like a function*). Crucially it **saves** what backward will need: `self.inp = inp` and `self.out = result`.
- **`backward(self)`** — uses the saved `self.inp` / `self.out` (and `self.out.g`, set by the next layer) to fill `self.inp.g` and any parameter gradients.

So the saved state is the bridge: forward stashes the values, backward reads them. The chain "save forward values, reuse in backward" from the earlier deep dive is now encapsulated *inside each layer*. A `Model` then just calls each layer forward in order, and `backward` in reverse — `for l in reversed(self.layers): l.backward()` — which is the chain rule expressed as a loop. This is precisely the structure PyTorch's `nn.Module` uses, as we'll confirm in Part 5.

In [ ]:
# ============================================================================
# RELU AS A CLASS
# ============================================================================
# This class wraps the ReLU activation function with forward and backward methods.

class Relu():
    """
    ReLU (Rectified Linear Unit) activation layer.
    
    Forward:  output = max(0, input)
    Backward: input.g = (input > 0) × output.g
    
    Attributes (set during forward pass):
        self.inp: The input tensor (saved for backward pass)
        self.out: The output tensor (saved for backward pass)
    """
    
    def __call__(self, inp):
        """
        Forward pass: Apply ReLU activation.
        
        Args:
            inp: Input tensor of any shape
        
        Returns:
            Output tensor with negative values set to 0
        
        The __call__ method makes the object callable like a function:
            relu_layer = Relu()
            output = relu_layer(input)  # Calls __call__ internally
        """
        # Save input for the backward pass
        # We need to know which values were positive to compute gradients
        self.inp = inp
        
        # Compute ReLU: max(0, x)
        # clamp_min(0.) sets all values below 0 to 0
        self.out = inp.clamp_min(0.)
        
        return self.out
    
    def backward(self):
        """
        Backward pass: Compute gradient of input.
        
        Formula: inp.g = (inp > 0) × out.g
        
        Intuition:
            - Where input was positive, gradient flows through unchanged
            - Where input was ≤ 0, gradient is blocked (ReLU was "off")
        
        Requires:
            self.out.g must be set (gradient from the next layer)
        
        Sets:
            self.inp.g: The gradient of the loss with respect to input
        """
        # (self.inp > 0) creates a boolean mask: True where inp was positive
        # .float() converts True/False to 1.0/0.0
        # Multiply by out.g: gradient passes through where mask is 1
        self.inp.g = (self.inp > 0).float() * self.out.g

print("Relu class defined!")
print("Usage: relu = Relu(); output = relu(input); relu.backward()")

In [ ]:
# ============================================================================
# LINEAR LAYER AS A CLASS
# ============================================================================
# This class wraps the linear transformation with its learnable parameters.

class Lin():
    """
    Linear (fully connected) layer.
    
    Forward:  output = input @ weight + bias
    Backward: Computes gradients for input, weight, and bias
    
    Attributes:
        self.w: Weight matrix (learnable parameter)
        self.b: Bias vector (learnable parameter)
        self.inp: Input tensor (saved during forward for backward)
        self.out: Output tensor (saved during forward for backward)
    """
    
    def __init__(self, w, b):
        """
        Initialize the linear layer with weights and biases.
        
        Args:
            w: Weight matrix, shape (input_features, output_features)
               Example: (784, 50) for 784 inputs → 50 outputs
            
            b: Bias vector, shape (output_features,)
               Example: (50,) for 50 outputs
        
        These are the LEARNABLE PARAMETERS that get updated during training.
        """
        self.w = w  # Store weight matrix
        self.b = b  # Store bias vector

    def __call__(self, inp):
        """
        Forward pass: Compute linear transformation.
        
        Args:
            inp: Input tensor, shape (batch_size, input_features)
        
        Returns:
            Output tensor, shape (batch_size, output_features)
        
        Formula: output = input @ weight + bias
        """
        # Save input for backward pass
        self.inp = inp
        
        # Compute linear transformation using our lin() function from earlier
        self.out = lin(inp, self.w, self.b)
        
        return self.out

    def backward(self):
        """
        Backward pass: Compute gradients for input, weights, and bias.
        
        Formulas:
            inp.g = out.g @ w.T      (gradient flows back to previous layer)
            w.g = inp.T @ out.g      (how much each weight contributed)
            b.g = sum(out.g)         (how much bias contributed)
        
        Requires:
            self.out.g must be set (gradient from the next layer)
        
        Sets:
            self.inp.g: Gradient for input (passed to previous layer)
            self.w.g: Gradient for weights (used for weight update)
            self.b.g: Gradient for bias (used for bias update)
        """
        # Gradient for input: distribute error back through weights
        self.inp.g = self.out.g @ self.w.t()
        
        # Gradient for weights: how much did each weight contribute to error?
        # inp.t() is transpose of input, making shapes compatible for matmul
        self.w.g = self.inp.t() @ self.out.g
        
        # Gradient for bias: sum across batch (bias affects all samples equally)
        self.b.g = self.out.g.sum(0)

print("Lin class defined!")
print("Usage: layer = Lin(w, b); output = layer(input); layer.backward()")

In [ ]:
c

In [ ]:
# ============================================================================
# MSE LOSS AS A CLASS
# ============================================================================
# This class wraps the Mean Squared Error loss function.

class Mse():
    """
    Mean Squared Error loss layer.
    
    Forward:  loss = mean((input - target)²)
    Backward: input.g = 2 × (input - target) / n
    
    This is used at the END of the network to compute how wrong we are.
    """
    
    def __call__(self, inp, targ):
        """
        Forward pass: Compute MSE loss.
        
        Args:
            inp:  Model predictions, shape (batch_size, 1)
            targ: True labels, shape (batch_size,)
        
        Returns:
            A single scalar: the mean squared error
        """
        # Save inputs for backward pass
        self.inp = inp
        self.targ = targ
        
        # Compute MSE using our mse() function from earlier
        self.out = mse(inp, targ)
        
        return self.out
    
    def backward(self):
        """
        Backward pass: Compute gradient of loss with respect to predictions.
        
        Formula: inp.g = 2 × (prediction - target) / n
        
        Where n = number of samples (for the mean)
        
        Derivation:
            loss = (1/n) × Σ(pred - targ)²
            d(loss)/d(pred) = (1/n) × 2 × (pred - targ)
                            = 2 × (pred - targ) / n
        
        Sets:
            self.inp.g: Gradient of loss with respect to input (predictions)
        """
        # self.inp.squeeze() converts (n, 1) → (n,) to match targ shape
        # Subtract target to get the error
        # Multiply by 2 (from derivative of x²)
        # Divide by n (from derivative of mean)
        # unsqueeze(-1) converts (n,) → (n, 1) to match inp shape
        self.inp.g = 2. * (self.inp.squeeze() - self.targ).unsqueeze(-1) / self.targ.shape[0]

print("Mse class defined!")
print("Usage: loss_fn = Mse(); loss = loss_fn(predictions, targets); loss_fn.backward()")

In [ ]:
# ============================================================================
# THE MODEL CLASS: COMBINING ALL LAYERS
# ============================================================================
# This class represents our complete neural network.
# It chains layers together and handles forward/backward for the whole network.

class Model():
    """
    A neural network model composed of multiple layers.
    
    Architecture:
        Input → Lin1 → ReLU → Lin2 → MSE Loss → Output
    
    The model handles:
        - Forward pass: chain all layers
        - Backward pass: reverse chain all layers
    """
    
    def __init__(self, w1, b1, w2, b2):
        """
        Initialize the model with weights and biases for both layers.
        
        Args:
            w1: Weights for first linear layer, shape (784, 50)
            b1: Biases for first linear layer, shape (50,)
            w2: Weights for second linear layer, shape (50, 1)
            b2: Biases for second linear layer, shape (1,)
        """
        # Create list of layers in forward order
        # This makes it easy to loop through them
        self.layers = [Lin(w1, b1), Relu(), Lin(w2, b2)]
        
        # Create loss function (separate from layers)
        self.loss = Mse()
        
    def __call__(self, x, targ):
        """
        Forward pass: compute predictions and loss.
        
        Args:
            x:    Input data, shape (batch_size, 784)
            targ: Target labels, shape (batch_size,)
        
        Returns:
            The MSE loss (a scalar)
        
        How it works:
            1. Pass data through each layer in sequence
            2. Compute loss between final output and targets
        """
        # Pass through each layer in sequence
        # x gets updated at each step: x → Lin1(x) → ReLU(x) → Lin2(x)
        for l in self.layers:
            x = l(x)
        
        # Compute and return loss
        return self.loss(x, targ)
    
    def backward(self):
        """
        Backward pass: compute all gradients.
        
        The key insight: we go through layers in REVERSE order!
        
        Order:
            1. Loss.backward()      → sets last layer's output gradient
            2. Lin2.backward()      → sets ReLU's output gradient
            3. ReLU.backward()      → sets Lin1's output gradient
            4. Lin1.backward()      → sets input gradient (rarely used)
        
        After this call:
            - All weight gradients (w1.g, w2.g) are computed
            - All bias gradients (b1.g, b2.g) are computed
        """
        # First, backprop through loss
        self.loss.backward()
        
        # Then, backprop through layers in REVERSE order
        # reversed() gives us [Lin2, Relu, Lin1]
        for l in reversed(self.layers):
            l.backward()

print("Model class defined!")
print("Usage:")
print("  model = Model(w1, b1, w2, b2)")
print("  loss = model(x_train, y_train)  # Forward pass")
print("  model.backward()                # Backward pass")

## How does `self.out.g` get set?  (the layered version: one layer's output **is** the next layer's input)

In the class version, `Lin.__call__` computes `self.out` but **never sets `self.out.g`** — yet `Lin.backward()` reads `self.out.g`. So who fills it in? **The *next* stage's `backward()` does** — because one layer's `self.out` is the *very same tensor object* as the next layer's `self.inp`.

### Why they are the same object

`Model.__call__` runs the forward pass as `x = l(x)` for each layer — each layer's **returned output is passed straight in as the next layer's input**, with no copy. So the same tensor flows down the chain wearing different names:

```
x0 ──Lin1──▶ x1 ──Relu──▶ x2 ──Lin2──▶ x3 ──Mse──▶ loss
              |             |            |
   Lin1.out = x1 = Relu.inp |            |
                  Relu.out = x2 = Lin2.inp
                               Lin2.out = x3 = Mse.inp
```

So three pairs are each literally **one object**:

- `Lin1.out` **is** `Relu.inp`
- `Relu.out` **is** `Lin2.inp`
- `Lin2.out` **is** `Mse.inp`

### Watch `Model.backward()` run in reverse

```python
self.loss.backward()              # 1
for l in reversed(self.layers):   # [Lin2, Relu, Lin1]
    l.backward()                  # 2, 3, 4
```

Writing `something.inp.g` mutates the shared object, so it shows up as the *previous* stage's `.out.g`:

| step | the code does | because of the shared object, this sets… |
|---|---|---|
| 1. `loss.backward()` | `self.inp.g = 2*(inp − targ)/n` | `Mse.inp` **is** `Lin2.out` → **`Lin2.out.g`** ✅ |
| 2. `Lin2.backward()` | reads `self.out.g`; `self.inp.g = out.g @ w.t()` | `Lin2.inp` **is** `Relu.out` → **`Relu.out.g`** ✅ |
| 3. `Relu.backward()` | reads `self.out.g`; `self.inp.g = (inp>0)*out.g` | `Relu.inp` **is** `Lin1.out` → **`Lin1.out.g`** ✅ |
| 4. `Lin1.backward()` | reads `self.out.g`; `self.inp.g = out.g @ w.t()` | sets the model-input gradient (done) |

Each layer's `self.out.g` was **deposited by the stage after it**, one moment earlier in the loop — into the very object this layer holds as `self.out`. So by the time a layer's `backward()` runs, its `self.out.g` is already there.

### The two facts that make it work

1. **The loss is the seed.** `Mse.backward()` needs *no* incoming `.g` — it computes `self.inp.g` directly from `2*(pred − targ)/n`. That first gradient, written onto `Mse.inp` (which *is* the last layer's `out`), kick-starts the whole chain.
2. **Setting an attribute mutates the shared object** (pass-by-object-reference — exactly the mechanism from the function version above). `next.inp.g = …` and `this.out.g` are *the same `.g` on the same tensor*, reached through two different names. Going in **reverse order** guarantees the writer (`next.backward()`) always runs before the reader (`this.backward()`).

> **In one line:** `self.out.g` is never produced inside the layer — the next stage's `backward()` writes its `self.inp.g`, and since that `inp` *is* the same tensor object as this layer's `out`, the gradient appears as `self.out.g`. The loss seeds the first one, and reverse order keeps the relay in sync. (`__call__` only had to do two things to enable this: pass its output straight on, **and** save `self.inp`/`self.out` so `backward()` can find those shared objects later.)

In [ ]:
# ============================================================================
# CREATE AN INSTANCE OF OUR MODEL
# ============================================================================
# Now let's create a model using our class-based architecture.
# We pass in the same weights we've been using all along.

model = Model(w1, b1, w2, b2)

print("Model created!")
print(f"Number of layers: {len(model.layers)}")
print(f"Layers: {[type(l).__name__ for l in model.layers]}")  # ['Lin', 'Relu', 'Lin']

In [ ]:
# ============================================================================
# FORWARD PASS USING THE MODEL CLASS
# ============================================================================
# Run the forward pass: input → layers → loss

loss = model(x_train, y_train)

print(f"Forward pass complete!")
print(f"Loss: {loss.item():.2f}")
print()
print("This should be the same as before - we're using the same weights!")

In [ ]:
# ============================================================================
# BACKWARD PASS USING THE MODEL CLASS
# ============================================================================
# Compute all gradients with a single call!

model.backward()

print("Backward pass complete!")
print()
print("Gradients have been computed for all parameters:")
print(f"  w1.g shape: {w1.g.shape}")
print(f"  b1.g shape: {b1.g.shape}")
print(f"  w2.g shape: {w2.g.shape}")
print(f"  b2.g shape: {b2.g.shape}")

In [ ]:
# ============================================================================
# VERIFY CLASS-BASED GRADIENTS MATCH OUR ORIGINAL IMPLEMENTATION
# ============================================================================
# The class-based model should produce identical gradients to our functional version.

# Compare with the gradients we saved earlier (from forward_and_backward)
test_close(w2g, w2.g, eps=0.01)  # Second layer weights
test_close(b2g, b2.g, eps=0.01)  # Second layer biases
test_close(w1g, w1.g, eps=0.01)  # First layer weights
test_close(b1g, b1.g, eps=0.01)  # First layer biases
test_close(ig, x_train.g, eps=0.01)  # Input gradients

print("✓ All gradient tests passed!")
print()
print("The class-based model produces identical gradients!")
print()
print("This proves our refactoring is correct - same math, cleaner code!")

---
## Part 4: The Module Pattern - A Better Abstraction

Our layer classes work, but there's repeated code in each one:
- `__call__` saves inputs and calls a computation method
- `backward` uses saved values

We can create a **base class** (`Module`) that handles the common patterns, then each layer just defines:
- `forward()`: The actual computation
- `bwd()`: The gradient computation

This is exactly how PyTorch's `nn.Module` works!

### Before vs After

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    BEFORE vs AFTER THE MODULE PATTERN                       │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   BEFORE (each class has boilerplate):                                      │
│   ────────────────────────────────────                                      │
│   class Relu():                                                             │
│       def __call__(self, inp):        # Repeated pattern                    │
│           self.inp = inp              # Save input                          │
│           self.out = inp.clamp_min(0.)  # Actual computation                │
│           return self.out                                                   │
│       def backward(self):                                                   │
│           self.inp.g = (self.inp>0).float() * self.out.g                   │
│                                                                             │
│   AFTER (using Module base class):                                          │
│   ────────────────────────────────                                          │
│   class Relu(Module):                                                       │
│       def forward(self, inp):         # Just the computation                │
│           return inp.clamp_min(0.)                                          │
│       def bwd(self, out, inp):        # Just the gradient                   │
│           inp.g = (inp>0).float() * out.g                                  │
│                                                                             │
│   The Module base class handles saving inputs and calling methods!          │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

# Deep Dive: factoring out a `Module` base class

---

Every layer in Part 3 repeated the same boilerplate: `__call__` saves the inputs, stores the output, and returns it; only the actual math differed. That repetition is a smell. Part 4 lifts the shared mechanics into a base class:

```python
class Module():
    def __call__(self, *args):
        self.args = args                 # save inputs for backward
        self.out  = self.forward(*args)  # subclass supplies the math
        return self.out
    def forward(self):  raise Exception('not implemented')
    def backward(self): self.bwd(self.out, *self.args)   # subclass supplies the grads
```

Now a concrete layer only writes the **two interesting pieces** — `forward` (the op) and `bwd` (its gradient) — and inherits the save/call/return plumbing:

```python
class Relu(Module):
    def forward(self, inp): return inp.clamp_min(0.)
    def bwd(self, out, inp): inp.g = (inp>0).float() * out.g
```

This is the **template-method pattern**: the base class fixes the *workflow* (save args → forward → later, backward via saved args), subclasses fill in the *steps*. The payoff is concrete in Part 5: PyTorch's real `nn.Module` is this same idea, just with autograd generating `bwd` for you so you only ever write `forward`.

In [ ]:
# ============================================================================
# THE MODULE BASE CLASS
# ============================================================================
# This base class provides the common functionality for all layers.
# Subclasses just need to implement forward() and bwd().

class Module():
    """
    Base class for all neural network layers.
    
    Provides:
        - Automatic saving of inputs during forward pass
        - Automatic retrieval of saved values during backward pass
        - Clean separation: subclasses just define forward() and bwd()
    
    Subclasses must implement:
        - forward(*args): The actual computation
        - bwd(out, *args): The gradient computation
    """
    
    def __call__(self, *args):
        """
        Callable interface for the forward pass.
        
        Args:
            *args: Variable arguments (e.g., input tensor, or multiple inputs)
                   The * means "accept any number of positional arguments"
        
        Returns:
            The output of the forward pass
        
        What this does:
            1. Saves all input arguments for the backward pass
            2. Calls the subclass's forward() method
            3. Saves and returns the output
        """
        # Save input arguments for backward pass
        self.args = args
        
        # Call the subclass's forward method with the inputs
        # *args unpacks the tuple: forward(args[0], args[1], ...)
        self.out = self.forward(*args)
        
        return self.out

    def forward(self):
        """
        Forward computation (to be implemented by subclass).
        
        Raises:
            Exception: Always, because this is meant to be overridden
        """
        raise Exception('not implemented')
    
    def backward(self):
        """
        Backward pass: compute gradients.
        
        Calls the subclass's bwd() method with:
            - self.out: The output from forward pass
            - *self.args: All the inputs from forward pass
        """
        # Call subclass's bwd with output and all saved inputs
        self.bwd(self.out, *self.args)
    
    def bwd(self):
        """
        Backward computation (to be implemented by subclass).
        
        Raises:
            Exception: Always, because this is meant to be overridden
        """
        raise Exception('not implemented')

print("Module base class defined!")
print("Subclasses inherit __call__ and backward, just implement forward and bwd.")

In [ ]:
# ============================================================================
# RELU USING THE MODULE PATTERN
# ============================================================================
# Much cleaner! We only define the essential logic.

class Relu(Module):
    """
    ReLU activation layer using Module pattern.
    
    Inherits from Module, so we just define:
        - forward(): The ReLU computation
        - bwd(): The gradient computation
    """
    
    def forward(self, inp):
        """
        Forward pass: Apply ReLU activation.
        
        Args:
            inp: Input tensor of any shape
        
        Returns:
            Input with negative values set to 0
        
        Note: We don't need to save inp manually - Module.__call__ does it!
        """
        return inp.clamp_min(0.)
    
    def bwd(self, out, inp):
        """
        Backward pass: Compute input gradient.
        
        Args:
            out: Output from forward pass (provided by Module.backward)
            inp: Input from forward pass (provided by Module.backward)
        
        Sets:
            inp.g: Gradient with respect to input
        """
        # Gradient flows through where input was positive
        inp.g = (inp > 0).float() * out.g

print("Relu(Module) class defined - much cleaner!")

In [ ]:
# ============================================================================
# LINEAR LAYER USING THE MODULE PATTERN
# ============================================================================

class Lin(Module):
    """
    Linear layer using Module pattern.
    
    Inherits from Module, adds __init__ for weights/biases.
    """
    
    def __init__(self, w, b):
        """
        Initialize with weight matrix and bias vector.
        
        Args:
            w: Weight matrix, shape (in_features, out_features)
            b: Bias vector, shape (out_features,)
        """
        self.w = w
        self.b = b
    
    def forward(self, inp):
        """
        Forward pass: Compute linear transformation.
        
        Args:
            inp: Input tensor, shape (batch_size, in_features)
        
        Returns:
            Output tensor, shape (batch_size, out_features)
        
        Formula: output = input @ weight + bias
        """
        return inp @ self.w + self.b
    
    def bwd(self, out, inp):
        """
        Backward pass: Compute gradients for input, weights, and bias.
        
        Args:
            out: Output from forward pass (has out.g set)
            inp: Input from forward pass
        
        Sets:
            inp.g: Gradient for input
            self.w.g: Gradient for weights
            self.b.g: Gradient for bias
        """
        # Gradient for input
        inp.g = self.out.g @ self.w.t()
        
        # Gradient for weights
        self.w.g = inp.t() @ self.out.g
        
        # Gradient for bias
        self.b.g = self.out.g.sum(0)

print("Lin(Module) class defined!")

In [ ]:
# ============================================================================
# MSE LOSS USING THE MODULE PATTERN
# ============================================================================

class Mse(Module):
    """
    MSE Loss layer using Module pattern.
    
    Note: This takes TWO arguments (predictions and targets).
    The Module base class handles multiple arguments via *args.
    """
    
    def forward(self, inp, targ):
        """
        Forward pass: Compute MSE loss.
        
        Args:
            inp:  Model predictions, shape (batch_size, 1)
            targ: True labels, shape (batch_size,)
        
        Returns:
            Scalar MSE loss
        """
        # squeeze() removes the trailing dimension: (n, 1) → (n,)
        # Then compute (pred - targ)² and take the mean
        return (inp.squeeze() - targ).pow(2).mean()
    
    def bwd(self, out, inp, targ):
        """
        Backward pass: Compute gradient of loss with respect to predictions.
        
        Args:
            out:  The loss value (output from forward)
            inp:  Predictions from forward pass
            targ: Targets from forward pass
        
        Sets:
            inp.g: Gradient of loss with respect to predictions
        """
        # d(MSE)/d(pred) = 2 × (pred - targ) / n
        # unsqueeze(-1) converts (n,) back to (n, 1) to match inp shape
        inp.g = 2 * (inp.squeeze() - targ).unsqueeze(-1) / targ.shape[0]

print("Mse(Module) class defined!")

In [ ]:
# ============================================================================
# TEST THE MODULE-BASED MODEL
# ============================================================================
# Create a new model using our Module-based layers.
# The Model class from earlier still works - it just uses the new layer classes!

model = Model(w1, b1, w2, b2)

print("Model created with Module-based layers!")
print(f"Layer types: {[type(l).__name__ for l in model.layers]}")

In [ ]:
# ============================================================================
# FORWARD PASS WITH MODULE-BASED MODEL
# ============================================================================

loss = model(x_train, y_train)

print(f"Forward pass complete! Loss: {loss.item():.2f}")

In [ ]:
# ============================================================================
# BACKWARD PASS WITH MODULE-BASED MODEL
# ============================================================================

model.backward()

print("Backward pass complete! Gradients computed.")

In [ ]:
# ============================================================================
# VERIFY MODULE-BASED GRADIENTS
# ============================================================================
# These should still match our original gradients!

test_close(w2g, w2.g, eps=0.01)
test_close(b2g, b2.g, eps=0.01)
test_close(w1g, w1.g, eps=0.01)
test_close(b1g, b1.g, eps=0.01)
test_close(ig, x_train.g, eps=0.01)

print("✓ All gradient tests passed!")
print()
print("The Module-based implementation produces identical gradients!")
print("We now have a clean, extensible architecture.")

---
## Part 5: Using PyTorch's nn.Module and Autograd

Now let's see how PyTorch does all of this for us! We'll:
1. Use `nn.Module` - PyTorch's base class for layers
2. Use autograd - automatic gradient computation
3. Use built-in layers - no manual gradient code needed!

### The Power of Autograd

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        WHAT WE DID vs WHAT PYTORCH DOES                     │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   OUR IMPLEMENTATION:                                                       │
│   ───────────────────                                                       │
│   1. Define forward computation manually                                    │
│   2. Derive gradient formulas mathematically                                │
│   3. Implement backward pass manually                                       │
│   4. Debug gradient code when things go wrong                               │
│                                                                             │
│   PYTORCH AUTOGRAD:                                                         │
│   ─────────────────                                                         │
│   1. Define forward computation                                             │
│   2. That's it! Call .backward() and gradients appear!                     │
│                                                                             │
│   PyTorch records all operations during forward pass and automatically     │
│   computes gradients using the chain rule!                                  │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

## How this maps onto real PyTorch

Everything we hand-built now has a one-to-one PyTorch counterpart — that's the punchline of the notebook:

| What we built by hand | PyTorch equivalent |
|---|---|
| `lin(x, w, b)` | `nn.Linear(in, out)` |
| our `Relu` layer | `nn.ReLU()` / `F.relu` |
| `Model` holding a list of layers | `nn.Sequential(...)` / an `nn.Module` |
| `mse(out, targ)` | `F.mse_loss(out, targ)` |
| our `backward()` chain | `loss.backward()` (autograd) |
| `w -= lr * w.g` | an optimizer step (`opt.step()`) |

The big difference: with autograd you **only write the forward pass**. PyTorch records the operations and derives every `backward` for you — exactly the `lin_grad` / ReLU-mask logic we wrote, generated automatically. You now know what it's doing under the hood.

In [ ]:
# ============================================================================
# IMPORT PYTORCH'S NEURAL NETWORK MODULES
# ============================================================================

from torch import nn               # nn contains all neural network components
import torch.nn.functional as F    # F contains functional versions (like mse_loss)

print("PyTorch nn module imported!")
print()
print("What we get:")
print("  - nn.Module: Base class for all layers (like our Module)")
print("  - nn.Linear: Built-in linear layer (like our Lin)")
print("  - nn.ReLU: Built-in ReLU activation (like our Relu)")
print("  - F.mse_loss: MSE loss function (like our Mse)")

In [ ]:
# ============================================================================
# CUSTOM LINEAR LAYER USING PyTorch nn.Module
# ============================================================================
# Let's create our own Linear layer using PyTorch's nn.Module.
# This shows how to create custom layers that work with autograd.

class Linear(nn.Module):
    """
    A linear layer using PyTorch's nn.Module.
    
    Unlike our manual implementation, we:
        - DON'T need to implement backward - autograd does it!
        - Use requires_grad_() to enable gradient tracking
        - Inherit from nn.Module for PyTorch integration
    """
    
    def __init__(self, n_in, n_out):
        """
        Initialize the linear layer.
        
        Args:
            n_in:  Number of input features (e.g., 784 for MNIST pixels)
            n_out: Number of output features (e.g., 50 for hidden layer)
        """
        # IMPORTANT: Must call parent's __init__
        # This registers the layer with PyTorch's module system
        super().__init__()
        
        # Create weight matrix with random values
        # requires_grad_() tells PyTorch to track operations for gradient computation
        self.w = torch.randn(n_in, n_out).requires_grad_()
        
        # Create bias vector initialized to zeros
        # Also enable gradient tracking
        self.b = torch.zeros(n_out).requires_grad_()
    
    def forward(self, inp):
        """
        Forward pass: Compute linear transformation.
        
        Args:
            inp: Input tensor, shape (batch_size, n_in)
        
        Returns:
            Output tensor, shape (batch_size, n_out)
        
        Note: We DON'T define backward! PyTorch autograd handles it!
        """
        return inp @ self.w + self.b

print("Linear(nn.Module) class defined!")
print("Notice: No backward method needed - autograd computes gradients automatically!")

In [ ]:
# ============================================================================
# MODEL USING PyTorch nn.Module
# ============================================================================
# A complete model using PyTorch's building blocks.

class Model(nn.Module):
    """
    A neural network model using PyTorch's nn.Module.
    
    Architecture:
        Linear(784→50) → ReLU → Linear(50→1)
    
    Key difference from our implementation:
        - No backward method needed!
        - PyTorch automatically computes all gradients!
    """
    
    def __init__(self, n_in, nh, n_out):
        """
        Initialize the model.
        
        Args:
            n_in:  Number of input features (784 for MNIST)
            nh:    Number of hidden neurons (50)
            n_out: Number of output features (1)
        """
        super().__init__()
        
        # Create layers as a list
        # Linear: our custom layer from above
        # nn.ReLU(): PyTorch's built-in ReLU activation
        self.layers = [Linear(n_in, nh), nn.ReLU(), Linear(nh, n_out)]
        
    def __call__(self, x, targ):
        """
        Forward pass and loss computation.
        
        Args:
            x:    Input data, shape (batch_size, n_in)
            targ: Target labels, shape (batch_size,)
        
        Returns:
            The MSE loss (a scalar with gradient tracking enabled)
        """
        # Pass through all layers
        for l in self.layers:
            x = l(x)
        
        # Compute MSE loss using PyTorch's built-in function
        # targ[:,None] converts (n,) → (n, 1) to match x's shape
        return F.mse_loss(x, targ[:, None])

print("Model(nn.Module) class defined!")
print("Uses PyTorch's autograd - no manual backward pass needed!")

In [ ]:
# ============================================================================
# TRAIN WITH PyTorch AUTOGRAD
# ============================================================================
# Let's use our PyTorch-based model!

# Create the model
# m = 784 (input features), nh = 50 (hidden), 1 (output)
model = Model(m, nh, 1)

# Forward pass: compute predictions and loss
# PyTorch builds a computation graph tracking all operations
loss = model(x_train, y_train)

# Backward pass: compute ALL gradients with one call!
# PyTorch traverses the computation graph in reverse
# and applies the chain rule automatically
loss.backward()

print(f"Loss: {loss.item():.2f}")
print()
print("Gradients computed automatically!")
print("No manual gradient formulas needed!")

In [ ]:
# ============================================================================
# EXAMINE THE AUTOGRAD GRADIENTS
# ============================================================================
# Let's look at the gradients PyTorch computed for our first layer.

# Get the first layer (our Linear class)
l0 = model.layers[0]

# Display the bias gradients
# Note: PyTorch stores gradients in .grad, not .g like we did
print("First layer bias gradients (l0.b.grad):")
print(l0.b.grad)
print()
print(f"Shape: {l0.b.grad.shape}")
print()
print("These gradients tell PyTorch how to adjust each bias to reduce loss.")
print("With these gradients, we could update weights: bias -= learning_rate * bias.grad")

---
## Summary: What We Learned

### The Complete Journey

We built backpropagation from scratch, step by step:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        OUR LEARNING JOURNEY                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  PART 1: Building Blocks                                                    │
│  ────────────────────────                                                   │
│  • Linear layer: output = input @ weights + bias                            │
│  • ReLU activation: output = max(0, input)                                  │
│  • MSE loss: loss = mean((prediction - target)²)                            │
│                                                                             │
│  PART 2: Backpropagation                                                    │
│  ────────────────────────                                                   │
│  • Chain rule: multiply derivatives along the path                          │
│  • Gradient formulas for each layer                                         │
│  • Verified against PyTorch autograd                                        │
│                                                                             │
│  PART 3: Object-Oriented Design                                             │
│  ────────────────────────────────                                           │
│  • Each layer as a class with forward() and backward()                      │
│  • Model chains layers together                                             │
│  • Cleaner, more extensible code                                            │
│                                                                             │
│  PART 4: Module Pattern                                                     │
│  ────────────────────────                                                   │
│  • Base class handles boilerplate                                           │
│  • Subclasses just define forward() and bwd()                               │
│  • Same pattern PyTorch uses!                                               │
│                                                                             │
│  PART 5: PyTorch Autograd                                                   │
│  ─────────────────────────                                                  │
│  • Define forward only - autograd handles backward!                         │
│  • .backward() computes all gradients automatically                         │
│  • Built-in layers: nn.Linear, nn.ReLU, etc.                                │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Key Takeaways

1. **Forward pass**: Data flows through layers, computing predictions
2. **Backward pass**: Gradients flow back, telling us how to improve
3. **Chain rule**: The mathematical foundation - multiply derivatives along paths
4. **Autograd**: PyTorch automates gradient computation, but now you understand how!

### What's Next?

In the next notebooks, we'll:
- Train the network by updating weights with gradients
- Use proper optimizers (SGD, Adam, etc.)
- Handle batching for efficiency
- Add more advanced layers (convolution, normalization, etc.)

**You now understand the core algorithm behind all of deep learning!**

### 🎮 Interactive: what the gradients are *for*

Backprop gives us every weight's gradient; **gradient descent** is how we use it: `w ← w − lr · gradient`. Pick a learning rate and step down the bowl. Too small crawls, too big overshoots and diverges — a preview of notebook 12's optimizers.

In [ ]:
# ============================================================================
# INTERACTIVE (embedded figure) -- run this cell. Self-contained iframe;
# works in Jupyter, Colab, and the exported HTML. Re-run to reset it.
# ============================================================================
from IPython.display import HTML
HTML('<iframe src="data:text/html;base64,PCFET0NUWVBFIGh0bWw+PGh0bWwgbGFuZz0iZW4iPjxoZWFkPjxtZXRhIGNoYXJzZXQ9IlVURi04Ij4KPG1ldGEgbmFtZT0idmlld3BvcnQiIGNvbnRlbnQ9IndpZHRoPWRldmljZS13aWR0aCwgaW5pdGlhbC1zY2FsZT0xLjAiPgo8dGl0bGU+R3JhZGllbnQgZGVzY2VudDwvdGl0bGU+CjxzdHlsZT4KICA6cm9vdHstLWluazojMjUzMjRhOy0tc29mdDojNWE2Yjg2Oy0tbGluZTojZGRlNWYyOy0taW46I2Y2OTIxZTstLW91dDojMTZhM2EzOy0tYWM6IzdiNWNkNjstLWJnOiNlZWYzZmI7fQogICp7Ym94LXNpemluZzpib3JkZXItYm94O31odG1sLGJvZHl7bWFyZ2luOjA7Zm9udC1mYW1pbHk6IlNlZ29lIFVJIixzeXN0ZW0tdWksc2Fucy1zZXJpZjtjb2xvcjp2YXIoLS1pbmspO30KICBib2R5e2JhY2tncm91bmQ6cmFkaWFsLWdyYWRpZW50KDkwMHB4IDQwMHB4IGF0IDMwJSAtMTAlLCNmM2Y3ZmYsdmFyKC0tYmcpKSx2YXIoLS1iZyk7cGFkZGluZzoxNHB4O30KICBoMXtmb250LXNpemU6MTdweDttYXJnaW46MCAwIDJweDt0ZXh0LWFsaWduOmNlbnRlcjt9CiAgLnN1Ynt0ZXh0LWFsaWduOmNlbnRlcjtjb2xvcjp2YXIoLS1zb2Z0KTtmb250LXNpemU6MTJweDttYXJnaW46MCAwIDEwcHg7fQogIC5jYXJke2JhY2tncm91bmQ6I2ZmZjtib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpO2JvcmRlci1yYWRpdXM6MTJweDtwYWRkaW5nOjEwcHg7Ym94LXNoYWRvdzowIDZweCAxNnB4IHJnYmEoMTIzLDkyLDIxNCwuMDgpO3dpZHRoOjUyMHB4O21heC13aWR0aDo5MnZ3O21hcmdpbjowIGF1dG87fQogIGNhbnZhc3tkaXNwbGF5OmJsb2NrO2JhY2tncm91bmQ6I2ZiZmRmZjtib3JkZXItcmFkaXVzOjhweDtib3JkZXI6MXB4IHNvbGlkIHZhcigtLWxpbmUpO3dpZHRoOjEwMCU7aGVpZ2h0OmF1dG87fQogIC5yb3d7ZGlzcGxheTpmbGV4O2dhcDo4cHg7YWxpZ24taXRlbXM6Y2VudGVyO2p1c3RpZnktY29udGVudDpjZW50ZXI7bWFyZ2luOjEwcHggMCAycHg7Zm9udC1zaXplOjEzcHg7ZmxleC13cmFwOndyYXA7fQogIGlucHV0W3R5cGU9cmFuZ2Vde3dpZHRoOjI0MHB4O21heC13aWR0aDo3MnZ3O2FjY2VudC1jb2xvcjp2YXIoLS1hYyk7fQogIC5jdHJ7ZGlzcGxheTpmbGV4O2dhcDo4cHg7anVzdGlmeS1jb250ZW50OmNlbnRlcjttYXJnaW4tdG9wOjhweDtmbGV4LXdyYXA6d3JhcDt9CiAgYnV0dG9ue2ZvbnQtZmFtaWx5OmluaGVyaXQ7Zm9udC13ZWlnaHQ6NzAwO2ZvbnQtc2l6ZToxM3B4O2JvcmRlcjpub25lO2JvcmRlci1yYWRpdXM6OXB4O3BhZGRpbmc6OHB4IDE1cHg7Y3Vyc29yOnBvaW50ZXI7fQogIC5zdGVwe2JhY2tncm91bmQ6dmFyKC0tYWMpO2NvbG9yOiNmZmY7fS5hdXRve2JhY2tncm91bmQ6dmFyKC0taW4pO2NvbG9yOiNmZmY7fS5yZXNldHtiYWNrZ3JvdW5kOiNlY2U3ZmI7Y29sb3I6dmFyKC0tYWMpO30KICAuc3RhdHVze3RleHQtYWxpZ246Y2VudGVyO2ZvbnQtZmFtaWx5Om1vbm9zcGFjZTtmb250LXNpemU6MTIuNXB4O21hcmdpbi10b3A6OHB4O30KICAub2t7Y29sb3I6dmFyKC0tb3V0KTtmb250LXdlaWdodDo3MDA7fS5iYWR7Y29sb3I6I2Q2NDU0NTtmb250LXdlaWdodDo3MDA7fQogIC5ub3Rle3RleHQtYWxpZ246Y2VudGVyO2ZvbnQtc2l6ZToxMS41cHg7Y29sb3I6dmFyKC0tc29mdCk7bWFyZ2luLXRvcDo4cHg7bWF4LXdpZHRoOjU2MHB4O21hcmdpbi1sZWZ0OmF1dG87bWFyZ2luLXJpZ2h0OmF1dG87bGluZS1oZWlnaHQ6MS41O30KPC9zdHlsZT48L2hlYWQ+PGJvZHk+CjxoMT5HcmFkaWVudCBkZXNjZW50OiBzbWFsbCBzdGVwcyBkb3duaGlsbDwvaDE+CjxwIGNsYXNzPSJzdWIiPlVwZGF0ZSBydWxlOiA8Yj53IOKGkCB3IOKIkiBsciDCtyBncmFkaWVudDwvYj4uIFBpY2sgYSBsZWFybmluZyByYXRlIGFuZCBzdGVwLiBUb28gc21hbGwg4oaSIGNyYXdsczsganVzdCByaWdodCDihpIgY29udmVyZ2VzOyB0b28gYmlnIOKGkiBvdmVyc2hvb3RzIGFuZCBkaXZlcmdlcy48L3A+CjxkaXYgY2xhc3M9ImNhcmQiPjxjYW52YXMgaWQ9ImN2IiB3aWR0aD0iNTIwIiBoZWlnaHQ9IjMwMCI+PC9jYW52YXM+PC9kaXY+CjxkaXYgY2xhc3M9InJvdyI+bGVhcm5pbmcgcmF0ZSA9IDxpbnB1dCBpZD0ibHIiIHR5cGU9InJhbmdlIiBtaW49IjAuMDEiIG1heD0iMS4wNSIgc3RlcD0iMC4wMSIgdmFsdWU9IjAuMSI+PHNwYW4gaWQ9ImxydiIgc3R5bGU9ImZvbnQtZmFtaWx5Om1vbm9zcGFjZTtjb2xvcjp2YXIoLS1hYyk7Zm9udC13ZWlnaHQ6NzAwIj4wLjEwPC9zcGFuPjwvZGl2Pgo8ZGl2IGNsYXNzPSJjdHIiPgogIDxidXR0b24gY2xhc3M9InN0ZXAiIGlkPSJzdGVwIj5TdGVwIOKGkzwvYnV0dG9uPgogIDxidXR0b24gY2xhc3M9ImF1dG8iIGlkPSJhdXRvIj7ij6kgQXV0bzwvYnV0dG9uPgogIDxidXR0b24gY2xhc3M9InJlc2V0IiBpZD0icmVzZXQiPuKGuiBSZXNldDwvYnV0dG9uPgo8L2Rpdj4KPGRpdiBjbGFzcz0ic3RhdHVzIiBpZD0ic3RhdHVzIj48L2Rpdj4KPHAgY2xhc3M9Im5vdGUiPkxvc3MgaGVyZSBpcyA8Y29kZT5MKHcpID0gd8KyPC9jb2RlPiwgc28gdGhlIGdyYWRpZW50IGlzIDxjb2RlPjJ3PC9jb2RlPiDigJQgZXhhY3RseSB0aGUga2luZCBvZiBzbG9wZSB0aGUgYmFja3dhcmQgcGFzcyBjb21wdXRlcyBmb3IgZXZlcnkgd2VpZ2h0LiBUaGUgb3B0aW1pemVyIGp1c3QgbnVkZ2VzIGVhY2ggd2VpZ2h0IGFnYWluc3QgaXRzIGdyYWRpZW50LCBvdmVyIGFuZCBvdmVyLjwvcD4KPHNjcmlwdD4KY29uc3QgJD1pZD0+ZG9jdW1lbnQuZ2V0RWxlbWVudEJ5SWQoaWQpOwpjb25zdCBMPXc9PncqdywgZEw9dz0+Mip3OwpsZXQgdz0yLjYsIGxyPTAuMSwgaGlzdD1bMi42XSwgdGltZXI9bnVsbDsKZnVuY3Rpb24gZHJhdygpewogIGNvbnN0IGN2PSQoJ2N2JyksY3R4PWN2LmdldENvbnRleHQoJzJkJyksVz1jdi53aWR0aCxIPWN2LmhlaWdodCxwYWQ9MzQ7CiAgY3R4LmNsZWFyUmVjdCgwLDAsVyxIKTsKICBjb25zdCB4bWluPS0zLHhtYXg9Myx5bWluPTAseW1heD05OwogIGNvbnN0IFg9dj0+cGFkKyh2LXhtaW4pLyh4bWF4LXhtaW4pKihXLTIqcGFkKTsKICBjb25zdCBZPXY9PkgtcGFkLSh2LXltaW4pLyh5bWF4LXltaW4pKihILTIqcGFkKTsKICBjdHguc3Ryb2tlU3R5bGU9JyNlM2U5ZjQnO2N0eC5saW5lV2lkdGg9MTtjdHguYmVnaW5QYXRoKCk7Y3R4Lm1vdmVUbyhwYWQsWSgwKSk7Y3R4LmxpbmVUbyhXLXBhZCxZKDApKTtjdHguc3Ryb2tlKCk7CiAgY3R4LmZpbGxTdHlsZT0nIzlmYjBjYyc7Y3R4LmZvbnQ9JzEwcHggbW9ub3NwYWNlJztjdHguZmlsbFRleHQoJ3cnLFctcGFkLTEwLFkoMCkrMTQpOwogIC8vIGJvd2wKICBjdHguc3Ryb2tlU3R5bGU9JyNjZGI4ZjUnO2N0eC5saW5lV2lkdGg9MztjdHguYmVnaW5QYXRoKCk7bGV0IHN0PWZhbHNlOwogIGZvcihsZXQgcHg9cGFkO3B4PD1XLXBhZDtweCsrKXtjb25zdCB4dj14bWluKyhweC1wYWQpLyhXLTIqcGFkKSooeG1heC14bWluKTtjb25zdCB5dj1MKHh2KTtpZih5dj55bWF4KXtzdD1mYWxzZTtjb250aW51ZTt9Y29uc3QgcHk9WSh5dik7aWYoIXN0KXtjdHgubW92ZVRvKHB4LHB5KTtzdD10cnVlO31lbHNlIGN0eC5saW5lVG8ocHgscHkpO30KICBjdHguc3Ryb2tlKCk7CiAgLy8gcGF0aAogIGN0eC5zdHJva2VTdHlsZT0ncmdiYSgyNDYsMTQ2LDMwLC40NSknO2N0eC5saW5lV2lkdGg9MjtjdHguYmVnaW5QYXRoKCk7CiAgaGlzdC5mb3JFYWNoKChodyxpKT0+e2NvbnN0IHB4PVgoTWF0aC5tYXgoeG1pbixNYXRoLm1pbih4bWF4LGh3KSkpLHB5PVkoTWF0aC5taW4oeW1heCxMKGh3KSkpO2lmKGk9PT0wKWN0eC5tb3ZlVG8ocHgscHkpO2Vsc2UgY3R4LmxpbmVUbyhweCxweSk7fSk7CiAgY3R4LnN0cm9rZSgpOwogIGhpc3QuZm9yRWFjaCgoaHcsaSk9PntpZihNYXRoLmFicyhodyk+eG1heClyZXR1cm47Y3R4LmZpbGxTdHlsZT1pPT09aGlzdC5sZW5ndGgtMT8nI2Y2OTIxZSc6J3JnYmEoMjQ2LDE0NiwzMCwuNDUpJztjdHguYmVnaW5QYXRoKCk7Y3R4LmFyYyhYKGh3KSxZKE1hdGgubWluKHltYXgsTChodykpKSxpPT09aGlzdC5sZW5ndGgtMT82OjMuNSwwLDcpO2N0eC5maWxsKCk7aWYoaT09PWhpc3QubGVuZ3RoLTEpe2N0eC5zdHJva2VTdHlsZT0nI2ZmZic7Y3R4LmxpbmVXaWR0aD0yO2N0eC5zdHJva2UoKTt9fSk7CiAgLy8gbWluIG1hcmtlcgogIGN0eC5maWxsU3R5bGU9JyMxNmEzYTMnO2N0eC5iZWdpblBhdGgoKTtjdHguYXJjKFgoMCksWSgwKSw0LDAsNyk7Y3R4LmZpbGwoKTsKfQpmdW5jdGlvbiBzdGF0dXMoKXsKICBjb25zdCBnPWRMKHcpLGRpdmVyZ2luZz1NYXRoLmFicyh3KT41OwogIGxldCBzPSdzdGVwICcrKGhpc3QubGVuZ3RoLTEpKycgwrcgdyA9ICcrdy50b0ZpeGVkKDMpKycgwrcgbG9zcyA9ICcrTCh3KS50b0ZpeGVkKDMpKycgwrcgZ3JhZCA9ICcrZy50b0ZpeGVkKDMpOwogIGlmKGRpdmVyZ2luZykgcys9JyAgPHNwYW4gY2xhc3M9ImJhZCI+4oaSIGRpdmVyZ2luZyEgbHIgdG9vIGJpZzwvc3Bhbj4nOwogIGVsc2UgaWYoTWF0aC5hYnModyk8MC4wMikgcys9JyAgPHNwYW4gY2xhc3M9Im9rIj7ihpIgY29udmVyZ2VkIOKckzwvc3Bhbj4nOwogICQoJ3N0YXR1cycpLmlubmVySFRNTD1zOwp9CmZ1bmN0aW9uIHN0ZXAoKXtjb25zdCBnPWRMKHcpO3c9dy1scipnO2hpc3QucHVzaCh3KTtpZihoaXN0Lmxlbmd0aD42MCloaXN0LnNoaWZ0KCk7ZHJhdygpO3N0YXR1cygpO2lmKE1hdGguYWJzKHcpPjUwKXtzdG9wQXV0bygpO319CmZ1bmN0aW9uIHN0b3BBdXRvKCl7Y2xlYXJJbnRlcnZhbCh0aW1lcik7dGltZXI9bnVsbDskKCdhdXRvJykudGV4dENvbnRlbnQ9J+KPqSBBdXRvJzt9CiQoJ2xyJykub25pbnB1dD1lPT57bHI9K2UudGFyZ2V0LnZhbHVlOyQoJ2xydicpLnRleHRDb250ZW50PWxyLnRvRml4ZWQoMik7fTsKJCgnc3RlcCcpLm9uY2xpY2s9KCk9PntzdG9wQXV0bygpO3N0ZXAoKTt9OwokKCdhdXRvJykub25jbGljaz0oKT0+e2lmKHRpbWVyKXtzdG9wQXV0bygpO3JldHVybjt9JCgnYXV0bycpLnRleHRDb250ZW50PSfij7ggUGF1c2UnO3RpbWVyPXNldEludGVydmFsKHN0ZXAsNDAwKTt9OwokKCdyZXNldCcpLm9uY2xpY2s9KCk9PntzdG9wQXV0bygpO3c9Mi42O2hpc3Q9WzIuNl07ZHJhdygpO3N0YXR1cygpO307CmRyYXcoKTtzdGF0dXMoKTsKPC9zY3JpcHQ+PC9ib2R5PjwvaHRtbD4K" style="width:100%; height:580px; border:1px solid #dde5f2; border-radius:12px; box-shadow:0 8px 24px rgba(123,92,214,.12);" loading="lazy" title="Gradient descent"></iframe>')


# Climate / EO bridge: backprop is the adjoint method

For readers coming from physical modelling, backpropagation is **reverse-mode automatic differentiation**, and it is mathematically the same machinery as the **adjoint method** used in geophysics:

- **Forward model** ↔ forward pass: integrate the model, *saving intermediate states*.
- **Adjoint model** ↔ backward pass: propagate sensitivities backward through those saved states, accumulating the gradient of a cost function w.r.t. every input/parameter in roughly one model's cost.
- **4D-Var data assimilation** and **parameter sensitivity studies** in climate/ocean models are gradient descent on a cost function whose gradient is obtained exactly this way.

So the `forward_and_backward` you just wrote by hand is, structurally, a tiny adjoint model — which is why differentiable physics and ML-for-climate emulators slot together so naturally.

---

## What's next

- **Notebook 04** wires this into a real training loop (mini-batches, an optimizer, metrics).
- **Notebook 11** fixes the crude `randn` initialisation we used here.
- **Notebook 12** replaces the plain `w -= lr*w.g` step with momentum/Adam — the gradient-descent figure above is the seed of all of it.

Once you've read through (especially the gradient deep dives — that's where to push back if anything feels off), run `concept-extraction` on this notebook to turn it into review cards.

In [ ]:
l0 = model.layers[0]
l0.b.grad

tensor([-19.60,  -2.40,  -0.12,   1.99,  12.78, -15.32, -18.45,   0.35,   3.75,  14.67,  10.81,  12.20,  -2.95, -28.33,
          0.76,  69.15, -21.86,  49.78,  -7.08,   1.45,  25.20,  11.27, -18.15, -13.13, -17.69, -10.42,  -0.13, -18.89,
        -34.81,  -0.84,  40.89,   4.45,  62.35,  31.70,  55.15,  45.13,   3.25,  12.75,  12.45,  -1.41,   4.55,  -6.02,
        -62.51,  -1.89,  -1.41,   7.00,   0.49,  18.72,  -4.84,  -6.52])